# Polymarket vs Kalshi: maker/maker — resting on both venues

The maker/maker-only rewrite of [02.2](02.2_cross_venue_confirmation_and_response.ipynb), built on [02.1](02.1_cross_venue_gaps_by_competition.ipynb) and blog part 2 ("What survives fees"). There, maker/maker was the only route positive after fees on most observations, in every group; the taker routes stay in 02.2. This notebook asks where in that route a strategy could start.

**Cells** are the four of 02.1 §2b, each at its selected freshness F: Premier League 2025/26 and Champions League 2025/26 in play (F = 60 s), World Cup 2026 in play (F = 5 s) and its last pregame hour (F = 10 s).

**Route.** `PM YES + K NO` rests a YES bid on Polymarket and a NO bid on Kalshi; it prices them from the last **PM taker-sell** print and one minus the last **K taker-buy** print. `K YES + PM NO` rests a YES bid on Kalshi and a NO bid on Polymarket, priced from the last **K taker-sell** and **PM taker-buy** prints. Reference net edge = `1 − PM cost − Kalshi cost − Kalshi maker fee` (Kalshi maker 1.75 % × p(1 − p), Polymarket maker 0; 02.1 §5). An observation is a print on either leg while the other leg is at most F old.

**Contents:** §1 price level by outcome and direction; §2 same-side gaps (buy/buy, sell/sell): §2.1 distribution, §2.2 by price level; §3 two-sided spread, fused vs each venue; §4 team win: PM only vs PM YES + K NO, paired; §5 team win: Hayashi–Yoshida lead–lag, PM bid vs Kalshi ask. §6 team win: waiting from one resting order's fill to the other side's next trade. §7 team win: replay simulation of the three resting orders; §7.1 with only two of them; §7.2 a buffer on the guard; §7.3 smaller orders; §7.4 a cash cap; §7.5 P&L drivers fixture by fixture; §7.6 per outcome vs per fixture, strong vs weak team; what the real-money test must measure. Planned: the first fill of a pair (anchor), the second fill (hedge).

**Inputs:** `conditions.parquet` and `matches.parquet` from [`gaps.py`](../gaps.py); `gap_cents.csv` and `gap_cents_price.csv` (`--by-price`) from [`gap_cents.py`](../gap_cents.py); `fused_spread.csv` and `pm_vs_cross.csv` (`--paired`) from [`fused_spread.py`](../fused_spread.py). `hy_lead_lag.parquet` from [`hy_lead_lag.py`](../hy_lead_lag.py). `leg_waits.parquet` from [`leg_waits.py`](../leg_waits.py). `mm_sim_*.parquet` from [`mm_sim.py`](../mm_sim.py); `mm_sim_L10_drivers.parquet` and `mm_sim_L10_jumps.parquet` from [`mm_drivers.py`](../mm_drivers.py) (`--size 50 / 20 / 15 / 10`: `mm_sim_L<size>_*`; with `--cap-pm/--cap-k`: `mm_sim_L10_cap200_300_*`) (§7.1: `mm_sim_legs_*`, `--variant legs`; §7.2: `mm_sim_buffer_*`, `--variant buffer`), which also reads the raw order books. Trade prints are references, not quotes: there are no books, queues or fills, and a positive reference edge assumes both resting legs fill.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research/soccer_1x2_analysis/ticks.py').exists())
ANALYSIS = ROOT / 'research/soccer_1x2_analysis'
sys.path.insert(0, str(ANALYSIS))
import gaps
from competition import BLUE, RED, GREEN, grid, finish

OUT = ANALYSIS / 'outputs/gap_analysis'
CFG = gaps.Config()
m = gaps.read_export(OUT / 'matches.parquet', ['match_id', 'league', 'season'])
m['combo'] = m.league.astype(str) + ' ' + m.season.astype(str) + '/' + (m.season % 100 + 1).astype(str).str.zfill(2)
COMBO = m.set_index('match_id').combo
FIXTURES = m.groupby('combo').size()
# the candidate cells of 02.1 §2b (70% dual-fresh rule), each at its selected F
CELLS = {'PL in play, F=60s': ('Premier League 2025/26', 'post_0_to_105m', 60),
         'UCL in play, F=60s': ('UEFA Champions League 2025/26', 'post_0_to_105m', 60),
         'WC in play, F=5s': ('World Cup 2026/27', 'post_0_to_105m', 5),
         'WC last pregame hour, F=10s': ('World Cup 2026/27', 'pre_last_hour', 10)}
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3, 'axes.titlesize': 9, 'font.size': 9})
print({cl: f'{FIXTURES[c[0]]} fixtures' for cl, c in CELLS.items()})

## 1. Price level by outcome and direction

Blog part 2 §4 pooled the three outcomes inside each price bucket. Part 2 §1 found the direction depends on the outcome: `PM YES + K NO` for team wins, `K YES + PM NO` for draws. Outcomes do not spread evenly over prices: draws trade mostly at 10–40 c, while team wins span the whole range. So the pooled price chart mixed an outcome effect with a price effect. Here each outcome is split out, in the direction part 2 picked for it:

- **team win** (home + away pooled): `PM YES + K NO`;
- **draw**: `K YES + PM NO`.

The other direction is drawn in grey beside each bar, for comparison. Without it there is no way to tell whether the chosen direction is the better one in a given bucket.

The bucket is the YES reference price on the venue holding YES: the PM sell print for `PM YES + K NO`, the K sell print for `K YES + PM NO`. Deciles are [lo, lo + 10) c. Three figures, each with rows = outcome and columns = cell:

1. **count**: fresh pair observations per bucket and per outcome, in thousands (team win = (home + away) / 2, so it compares with the single draw outcome);
2. **positive rate**: % of that bucket's observations with reference net edge > 0;
3. **mean net edge**: cents per observation after the Kalshi maker fee. This tells a 0.1 c overlap apart from a 3 c one, which the positive rate cannot do.

**Exclusion rule.** A bucket (cell × outcome × direction × price) is excluded if it fails any one of three thresholds: fewer than 10k observations per outcome, positive rate below 60 %, or mean net edge below 0.5 c. Excluded buckets are drawn light and hatched in all three figures, so one bucket looks the same everywhere. The buckets that pass all three are listed in the table after the figures. The net-edge axis is clipped at −1.5 c, because Champions League draws at 80–90 c are 2 observations at −31 c. The tables give observations, fixtures, positive rate and mean net edge for every bucket.

**Fees cannot separate the two directions.** At a given bucket price p, both directions pay the Kalshi maker fee on the same p(1 − p): a NO leg costs 1 − p, and the curve is symmetric. That fee is at most 0.44 c per contract (table below). So any gap between the two directions within a bucket is gross edge, meaning where each venue's prices sit, and not fees.

In [ ]:
COLS = ['route', 'direction', 'outcome', 'phase', 'freshness', 'price_bucket', 'n', 'pos_0c', 'edge_sum', 'match_id']
cd = gaps.read_export(OUT / 'conditions.parquet', COLS)
cd = cd[cd.route == 'maker/maker']
cd['combo'] = cd.match_id.map(COMBO)
cd = pd.concat([cd[(cd.combo == c) & (cd.phase == ph) & (cd.freshness == F)].assign(cell=cl) for cl, (c, ph, F) in CELLS.items()], ignore_index=True)
cd['sel'] = np.where(cd.outcome.astype(str) == 'draw', 'draw', 'team win')
cd['direction'] = cd.direction.astype(str)
PB = list(gaps.PRICE_BUCKETS)
SEL = {'team win': ('PM YES + K NO', 'K YES + PM NO', BLUE), 'draw': ('K YES + PM NO', 'PM YES + K NO', RED)}   # studied, comparison, colour
GREY = '#9e9e9e'
MIN_N_K, MIN_POS, MIN_NET = 10, 60, 0.5   # excluded if any fails: thousand observations per outcome, % positive, cents

g = cd.groupby(['cell', 'sel', 'direction', 'price_bucket'], observed=True).agg(
    n=('n', 'sum'), pos=('pos_0c', 'sum'), edge=('edge_sum', 'sum'), fixtures=('match_id', 'nunique')).reset_index()
g['price_bucket'] = g.price_bucket.astype(str)
g['n_k'] = g.n / 1000 / np.where(g.sel == 'team win', 2, 1)   # per outcome: team win pools home + away
g['pos_pct'] = 100 * g.pos / g.n
g['net_c'] = 100 * g.edge / g.n
g['studied'] = [SEL[s][0] == d for s, d in zip(g.sel, g.direction)]
g['excluded'] = (g.n_k < MIN_N_K) | (g.pos_pct < MIN_POS) | (g.net_c < MIN_NET)

def price_panels(col, ylabel, title, sharey):
    fig, axes = grid(len(SEL), len(CELLS), w=4.6, h=3.0, sharex=True, sharey=sharey)
    x, w = np.arange(len(PB)), .4
    for i, (sel, (studied, other, color)) in enumerate(SEL.items()):
        for j, cl in enumerate(CELLS):
            ax = axes[i, j]
            for k, (d, c, lab) in enumerate([(studied, color, studied), (other, GREY, f'{other} (comparison)')]):
                q = g[(g.cell == cl) & (g.sel == sel) & (g.direction == d)].set_index('price_bucket').reindex(PB)
                for b, ex in zip(ax.bar(x + (k - .5) * w, q[col], w, color=c, label=lab), q.excluded.fillna(False).astype(bool)):
                    if ex:
                        b.set_hatch('////'); b.set_alpha(.45)
            if col == 'net_c': ax.axhline(0, color='k', lw=.6)
            ax.set_xticks(x); ax.set_xticklabels(PB, rotation=45, fontsize=7)
        axes[i, 0].legend(fontsize=7, loc='upper center', ncol=2)
    if col == 'net_c':
        axes[0, 0].set_ylim(bottom=max(axes[0, 0].get_ylim()[0], -1.5))   # one 2-observation bucket sits at -31 c
    for ax in ([axes[0, 0]] if sharey else axes.flat):   # headroom for the legend, once per shared axis
        ax.set_ylim(top=ax.get_ylim()[1] * 1.25)
    for ax in axes[-1]: ax.set_xlabel('YES price on the venue holding YES')
    finish(fig, axes, [f'{s}: {SEL[s][0]}' for s in SEL], list(CELLS), ylabel, title)

price_panels('n_k', 'observations per outcome (thousands)', 'Maker/maker fresh pair observations by YES price level (hatched: excluded)', sharey=True)
price_panels('pos_pct', '% positive after fees', 'Maker/maker positive reference edge by YES price level (hatched: excluded)', sharey=True)
price_panels('net_c', 'mean net edge (cents)', 'Maker/maker mean net reference edge by YES price level (hatched: excluded; axis clipped at −1.5 c)', sharey=True)

keep = g[~g.excluded].copy()
keep['cell'] = pd.Categorical(keep.cell, list(CELLS), ordered=True)
keep['price_bucket'] = pd.Categorical(keep.price_bucket, PB, ordered=True)
print(f'Buckets passing all three thresholds (≥ {MIN_N_K}k observations per outcome, ≥ {MIN_POS} % positive, ≥ {MIN_NET} c mean net edge): {len(keep)} of {len(g)}')
with pd.option_context('display.max_rows', None):
    display(keep.sort_values(['cell', 'sel', 'direction', 'price_bucket']).set_index(['cell', 'sel', 'direction', 'price_bucket'])[['n_k', 'fixtures', 'pos_pct', 'net_c']].round(2))

mid = np.arange(.05, 1, .1)
print('Kalshi maker fee per contract at the bucket midpoint (cents), the same for both directions:')
display(pd.DataFrame([100 * gaps.kalshi_fee(mid, CFG.lot, CFG.kalshi_maker_rate) / CFG.lot], columns=PB, index=['fee_c']).round(2))
tot = g.groupby(['sel', 'direction', 'cell'])[['n', 'pos', 'edge']].sum()
tot = tot.assign(n_k=tot.n / 1000 / np.where(tot.index.get_level_values('sel') == 'team win', 2, 1), pos_pct=100 * tot.pos / tot.n, net_c=100 * tot.edge / tot.n)[['n_k', 'pos_pct', 'net_c']]
print('All buckets together (n_k per outcome; rates match 02.1 §5 maker/maker by outcome):')
display(tot.unstack('cell').reindex(columns=list(CELLS), level=1).round(2))
for col, name in [('n_k', 'observations per outcome (thousands)'), ('fixtures', 'fixtures'), ('pos_pct', '% positive'), ('net_c', 'mean net edge (cents)')]:
    print(f'--- {name}')
    t = g.pivot_table(index=['sel', 'direction', 'cell'], columns='price_bucket', values=col).reindex(columns=PB)
    display(t.reindex(pd.MultiIndex.from_tuples([(s, d, cl) for s, (a, b, _) in SEL.items() for d in (a, b) for cl in CELLS])).round(1))

**Reading the current export.**

*Fees are not the story.* The Kalshi maker fee is at most 0.44 c per contract at 50 c, and both directions pay it at the same price. Yet within one bucket the two directions differ by more than a cent: Premier League draws at 60–70 c, 0.2 c against 1.7 c; team wins at 50–60 c, 0.3 c against 1.2 c. Blog part 2 §4 blamed the fee curve for the mid-price drop of `K YES + PM NO`. The drop is gross edge: at mid prices, Polymarket's team-win prices sit low enough that Polymarket's ask barely clears Kalshi's bid.

*Team win, `PM YES + K NO`: flat and robust.*
- Premier League: 75–87 % positive in every bucket, mean net edge 1.0–1.4 c.
- Champions League: 75–88 %, 1.1–1.5 c (0.5 c at 90–100 c).
- World Cup in play: 74–88 %, 0.6–0.9 c.
- World Cup last pregame hour: 85–95 % in every bucket except 90–100 c, which rests on 3 fixtures.

The advantage over `K YES + PM NO` sits at **40–80 c**. There, in the Premier League and the World Cup in play, `K YES + PM NO` is at 52–60 % positive and 0.1–0.5 c, against 79–87 % and 0.8–1.4 c. Below 30 c the two directions' mean net edges are within 0.4 c of each other in every cell. `K YES + PM NO` is even slightly ahead at 0–20 c in the Premier League: 1.6 c against 1.2 c at 0–10 c.

*Draw, `K YES + PM NO`: right below 30 c in the club cells; the other direction wins from 40–50 c.*
- Premier League: 88 % positive and 2.2 c at 0–10 c; 85 % and 2.3 c at 10–20 c; 81 % and 1.5 c at 20–30 c. Then it falls to 55–60 % and 0.2–0.4 c at 50–70 c.
- Premier League, `PM YES + K NO` on draws: the reverse, rising from 66–72 % and 0.9–1.1 c below 30 c to 88–90 % and 1.5–1.7 c at 50–70 c. The two are level at 30–50 c.
- Champions League: the same crossover, a bucket later. At 30–40 c `K YES + PM NO` still leads (1.2 c against 0.8 c); at 40–50 c it trails (74 % and 0.7 c against 86 % and 1.2 c).
- World Cup in play: `K YES + PM NO` wins in every bucket with at least 10 fixtures, 0.5–0.9 c against 0.2–0.6 c.
- World Cup last pregame hour: draws trade only at 0–50 c. The picture is mixed there: `K YES + PM NO` leads at 20–40 c, the other direction at 10–20 c.

*The better price comes with the scarcer flow.* In each outcome, `PM YES + K NO` has about twice the observations of `K YES + PM NO` in the club cells:
- team wins (per outcome): 344k against 181k (PL), 202k against 111k (UCL);
- draws: 168k against 82k (PL), 96k against 47k (UCL).

The World Cup in play is closer: 376k against 288k per team-win outcome, 286k against 224k for draws. `K YES + PM NO` reads the Kalshi taker-**sell** print, and Kalshi's flow is buy-heavy (02.1 §4). The resting Kalshi YES bid of that direction is filled only by the scarce side.

The richest bars in the whole table are club draws below 20 c in `K YES + PM NO`: 2.2–2.8 c and 85–90 % positive. They are also thin: 27k Premier League observations, against 89k per team-win outcome in `PM YES + K NO` at the same prices.

**Implication for a strategy.** The quoting direction should be chosen by outcome **and** price, not by outcome alone:
- team win ≥ 30 c → `PM YES + K NO`; below 30 c the two directions are equivalent on mean net edge;
- club draw < 30 c → `K YES + PM NO`; club draw ≥ 40–50 c → `PM YES + K NO`;
- World Cup in-play draw → `K YES + PM NO` at every price.

Whether the scarce Kalshi-sell fill arrives often enough, and fast enough, to make the `K YES + PM NO` bars usable is the question for the anchor and hedge sessions.

## 2. Same-side gaps

### 2.1 Where each venue's ask and bid sit

§1 compares a bid-side print on one venue with an ask-side print on the other: that is the edge of a resting pair. Where to place each resting order is a different question: **on the same side of the book, which venue sits higher?** 02.1 §4 gave only the means (team win buy/buy about −0.3 c, draw sell/sell +0.15 to +0.5 c). Here are the distributions, drawn like 02.1 §3:

- **buy/buy**: PM taker-buy print − K taker-buy print. Both are ask-side trades. PM − K < 0 means Polymarket's ask trades lower, so a YES ask resting on Kalshi at Polymarket's level would be below Kalshi's usual ask-side prints.
- **sell/sell**: PM taker-sell − K taker-sell. Both are bid-side trades. PM − K > 0 means Polymarket's bid trades higher, so a YES bid resting on Kalshi at Polymarket's level would be above Kalshi's usual bid-side prints.

Same population as 02.1 §3: prints on either leg while the other is fresh at the cell's F. Dark bars count prints, light bars count fresh seconds. Buckets are signed whole cents: `(−1, 1)` holds sub-cent differences, and `(−2, −1]` / `[1, 2)` hold exact one-tick gaps. A one-tick level offset between the books shows up as an imbalance between those two buckets. A mean offset that sits inside `(−1, 1)` instead comes from same-second VWAP: a taker sweeping several levels on one venue raises its print without the resting level being different.

In [ ]:
cents = pd.read_csv(OUT / 'gap_cents.csv')
cents['combo'] = cents.league + ' ' + cents.season.astype(str) + '/' + (cents.season % 100 + 1).astype(str).str.zfill(2)
cents = pd.concat([cents[(cents.combo == c) & (cents.phase == ph) & (cents.freshness == F)].assign(cell=cl) for cl, (c, ph, F) in CELLS.items()], ignore_index=True)
cents['sel'] = np.where(cents.outcome == 'draw', 'draw', 'team win')
C = list(gaps.GAP_CENTS)
CENT_LABELS = [{-4: '≤−4', -3: '(−4,−3]', -2: '(−3,−2]', -1: '(−2,−1]', 0: '(−1,1)', 1: '[1,2)', 2: '[2,3)', 3: '[3,4)', 4: '≥4'}[c] for c in C]
SAME = {'buy/buy': 'ask side', 'sell/sell': 'bid side'}
SHADES = {'team win': (BLUE, '#b3b8ff'), 'draw': (RED, '#f7b2a6')}
ROWS = [(p, s) for p in SAME for s in SHADES]

sg = cents[cents.pair.isin(list(SAME))].groupby(['cell', 'pair', 'sel', 'cents'])[['n', 'time', 'gap_sum', 'gap_time_sum']].sum()
fig, axes = grid(len(ROWS), len(CELLS), w=4.6, h=2.8, sharex=True, sharey=True)
x, w = np.arange(len(C)), .4
stats = []
for i, (pair, sel) in enumerate(ROWS):
    for j, cl in enumerate(CELLS):
        d = sg.loc[(cl, pair, sel)].reindex(C, fill_value=0)
        pct_n, pct_t = 100 * d.n / d.n.sum(), 100 * d.time / d.time.sum()
        dark, light = SHADES[sel]
        axes[i, j].bar(x - w / 2, pct_n, w, color=dark, label='per print')
        axes[i, j].bar(x + w / 2, pct_t, w, color=light, label='per fresh second')
        axes[i, j].set_xticks(x); axes[i, j].set_xticklabels(CENT_LABELS, rotation=45, fontsize=7)
        axes[i, j].set_title(f'n={int(d.n.sum()):,}', fontsize=8)
        stats.append(dict(pair=pair, sel=sel, cell=cl, prints=int(d.n.sum()), mean_gap_c=d.gap_sum.sum() / d.n.sum(),
                          time_mean_gap_c=d.gap_time_sum.sum() / d.time.sum(),
                          le_m1c_pct=pct_n[[c for c in C if c <= -1]].sum(), sub_cent_pct=pct_n[0], ge_p1c_pct=pct_n[[c for c in C if c >= 1]].sum(),
                          one_tick_below_pct=pct_n[-1], one_tick_above_pct=pct_n[1]))
    axes[i, 0].legend(fontsize=7)
for ax in axes[-1]: ax.set_xlabel('PM − K (cents)')
finish(fig, axes, [f'{p} ({SAME[p]}) | {s}' for p, s in ROWS], list(CELLS), '% of fresh prints / seconds', 'Same-side cross-venue gap PM − K: buy/buy (ask side) and sell/sell (bid side)')
stats = pd.DataFrame(stats).set_index(['pair', 'sel', 'cell'])
print('Per print unless marked; one_tick_* = the (−2,−1] and [1,2) buckets:')
display(stats.round(2))

**Reading the current export.**

*Team win, ask side (buy/buy): a real one-tick offset.* Polymarket's ask-side print is exactly one tick below Kalshi's (the `(−2,−1]` bucket) on 22–29 % of fresh prints. It is one tick above on only 2–9 %. The mean is −0.27 to −0.36 c in every cell, with the per-second mean close behind. This is a level difference between the two books, not same-second VWAP. It is the same fact as §1's direction result for team wins. Kalshi's YES ask usually sits a tick above Polymarket's, and a Kalshi NO bid is a Kalshi YES ask, so `PM YES + K NO` collects that tick.

*Draw, bid side (sell/sell): the mirror image, weaker.* Polymarket's bid-side print is one tick above Kalshi's on 20–26 % of prints and one tick below on 10–16 %. The mean is +0.15 to +0.50 c. The imbalance is largest in the World Cup's last pregame hour (26 % against 10 %), but that cell has only 10k prints. A Kalshi YES bid posted at Kalshi's own level is therefore usually a tick cheaper than Polymarket's. That is the `K YES + PM NO` draw edge of §1.

*The other two sides are roughly balanced, with exceptions.*
- Draw buy/buy: the mean is near 0 (−0.08 to +0.13 c), although the one-tick buckets still lean slightly towards Polymarket lower (12–17 % against 5–12 %).
- Team-win sell/sell: balanced in play (one tick below 18–20 %, above 15–16 %).
- Team-win sell/sell in the World Cup's last pregame hour: Polymarket's bid is lower (29 % against 10 %).

*Dispersion.* The bid side is noisier than the ask side: only 37–61 % of sell/sell prints fall within a cent, against 54–82 % for buy/buy. Kalshi's sell prints are the scarce side (02.1 §2c), so they are older when paired.

**For placing resting orders.** On the side where one venue's book sits a tick higher, the choice is the fill-versus-edge trade-off:
- Post the Kalshi order at Polymarket's level (team-win YES ask, draw YES bid). It becomes the best price on Kalshi and fills more often, but gives up the tick.
- Post it at Kalshi's own level. It keeps the tick, but waits in the queue and fills mainly when the price moves through it, which is adverse selection.

Trade prints alone cannot measure this: they show where trades happened, not queue lengths or which orders were resting. The anchor session has to price both placements.

### 2.2 Same-side gaps by price level

The §2.1 gaps split by YES price decile, drawn like the three figures of §1. **These are raw trade-price gaps, without fees.** A same-side gap is not a route and pays no fee. It says where one venue's ask (or bid) trades relative to the other's. The fee-adjusted edge of a resting pair is §1.

The bucket is the midpoint of the two prints' YES prices, so neither venue defines it. Rows = outcome, columns = cell (each at its F). In each panel buy/buy (ask side) is dark and sell/sell (bid side) is light. The three figures:

1. **count**: fresh prints per bucket and per outcome, in thousands (team win = (home + away) / 2);
2. **one-tick imbalance**: % of prints with PM − K ≥ +1 c drawn above zero, and % with PM − K ≤ −1 c drawn below zero. The ask-side offset of §2.1 shows as a long bar below zero for team-win buy/buy; the draw bid-side offset as a long bar above zero for draw sell/sell;
3. **mean gap PM − K** in cents per print.

Rates of buckets with fewer than 10k prints per outcome are hatched, as in §1's first threshold.

In [ ]:
gp = pd.read_csv(OUT / 'gap_cents_price.csv')
gp['combo'] = gp.league + ' ' + gp.season.astype(str) + '/' + (gp.season % 100 + 1).astype(str).str.zfill(2)
gp = pd.concat([gp[(gp.combo == c) & (gp.phase == ph) & (gp.freshness == F)].assign(cell=cl) for cl, (c, ph, F) in CELLS.items()], ignore_index=True)
gp['sel'] = np.where(gp.outcome == 'draw', 'draw', 'team win')
gp['n_hi'] = gp.n.where(gp.cents >= 1, 0)
gp['n_lo'] = gp.n.where(gp.cents <= -1, 0)
pg = gp.groupby(['cell', 'sel', 'pair', 'price_bucket'])[['n', 'n_hi', 'n_lo', 'gap_sum']].sum().reset_index()
pg['n_k'] = pg.n / 1000 / np.where(pg.sel == 'team win', 2, 1)   # per outcome
pg['hi_pct'] = 100 * pg.n_hi / pg.n
pg['lo_pct'] = 100 * pg.n_lo / pg.n
pg['mean_gap_c'] = pg.gap_sum / pg.n
pg['thin'] = pg.n_k < MIN_N_K
# consistency with §2.1 (gap_cents.csv): the price split must partition the same prints
chk = pg.groupby(['cell', 'sel', 'pair'])[['n', 'gap_sum']].sum().join(sg.groupby(level=['cell', 'pair', 'sel'])[['n', 'gap_sum']].sum().reorder_levels(['cell', 'sel', 'pair']), rsuffix='_21')
assert np.allclose(chk.n, chk.n_21) and np.allclose(chk.gap_sum, chk.gap_sum_21), 'gap_cents_price.csv disagrees with gap_cents.csv'

def same_side_panels(kind, ylabel, title):
    fig, axes = grid(len(SHADES), len(CELLS), w=4.6, h=3.0, sharex=True, sharey=True)
    x, w = np.arange(len(PB)), .4
    for i, (sel, (dark, light)) in enumerate(SHADES.items()):
        for j, cl in enumerate(CELLS):
            ax = axes[i, j]
            for k, (pair, c) in enumerate([('buy/buy', dark), ('sell/sell', light)]):
                q = pg[(pg.cell == cl) & (pg.sel == sel) & (pg.pair == pair)].set_index('price_bucket').reindex(PB)
                thin = q.thin.fillna(False).astype(bool)
                lab = f'{pair} ({SAME[pair]})'
                if kind == 'imbalance':
                    bs = list(ax.bar(x + (k - .5) * w, q.hi_pct, w, color=c, label=lab)) + list(ax.bar(x + (k - .5) * w, -q.lo_pct, w, color=c))
                    flags = list(thin) * 2
                else:
                    bs, flags = ax.bar(x + (k - .5) * w, q[kind], w, color=c, label=lab), thin
                for b, f in zip(bs, flags):
                    if f and kind != 'n_k':
                        b.set_hatch('////'); b.set_alpha(.45)
            if kind != 'n_k': ax.axhline(0, color='k', lw=.6)
            ax.set_xticks(x); ax.set_xticklabels(PB, rotation=45, fontsize=7)
        axes[i, 0].legend(fontsize=7, loc='upper center', ncol=2)
    lo, hi = axes[0, 0].get_ylim()
    axes[0, 0].set_ylim(lo, hi + (hi - lo) * .25)   # headroom for the legend
    for ax in axes[-1]: ax.set_xlabel('YES price, midpoint of the two prints')
    finish(fig, axes, list(SHADES), list(CELLS), ylabel, title)

same_side_panels('n_k', 'prints per outcome (thousands)', 'Same-side fresh prints by YES price level')
same_side_panels('imbalance', '% ≥ +1 c (up) | % ≤ −1 c (down)', 'Same-side one-tick imbalance PM − K by YES price level, no fees (hatched: < 10k prints)')
same_side_panels('mean_gap_c', 'mean PM − K (cents)', 'Same-side mean gap PM − K by YES price level, no fees (hatched: < 10k prints)')
for col, name in [('n_k', 'prints per outcome (thousands)'), ('hi_pct', '% PM − K ≥ +1 c'), ('lo_pct', '% PM − K ≤ −1 c'), ('mean_gap_c', 'mean PM − K (cents)')]:
    print(f'--- {name}')
    t = pg.pivot_table(index=['pair', 'sel', 'cell'], columns='price_bucket', values=col).reindex(columns=PB)
    display(t.reindex(pd.MultiIndex.from_tuples([(p, s, cl) for p in SAME for s in SHADES for cl in CELLS])).round(2))

**Reading the current export** (means per print, no fees).

*In the club cells the offset follows the price, on both sides of the book.* Polymarket trades **higher** than Kalshi on cheap outcomes and **lower** on expensive ones, for team wins and draws alike:
- Premier League team wins: sell/sell is +0.2 to +0.3 c below 30 c and −0.3 to −0.5 c at 40–90 c. buy/buy is −0.03 to −0.14 c below 30 c and −0.5 to −0.7 c at 40–90 c.
- Premier League draws: buy/buy is +0.24 to +0.29 c below 20 c and −0.4 to −0.8 c at 40–70 c. sell/sell is +0.4 to +0.7 c below 40 c and −0.7 c at 60–70 c.
- The Champions League has the same shape, weaker on team wins.

Two of §2.1's "not obvious" pooled means are cancellations, not absences. Team-win sell/sell and draw buy/buy are each positive at low prices and negative at high prices. They average out only because of where each outcome's prints sit.

*The World Cup differs.*
- In play, draws trade higher on Polymarket at almost every price, on both sides: sell/sell +0.25 to +0.38 c, buy/buy up to +0.24 c.
- In play, the team-win ask side is lower at every price (−0.1 to −0.5 c).
- In the last pregame hour, Polymarket is lower on both sides of team wins at almost every price.

*Link to §1.* This is where §1's direction crossover comes from. Where Polymarket trades lower (club outcomes above about 40 c, World Cup team wins), buying YES on Polymarket and NO on Kalshi collects the offset: `PM YES + K NO`. Where Polymarket trades higher (club outcomes below about 30 c, World Cup draws), the reverse does: `K YES + PM NO`. The offset in cents is small next to a one-cent tick. It is a shift in how often the two books sit a tick apart, not a whole-tick level difference at every moment.

**For placing resting orders.** The side on which the Kalshi order should shade away from Polymarket's level depends on the price band as well as the outcome:
- Club, above about 40 c: Polymarket is lower on both sides. The Kalshi YES ask can sit a tick above Polymarket's ask (the `PM YES + K NO` hedge leg).
- Club, below about 30 c: Polymarket is higher on both sides. The Kalshi YES bid can sit a tick below Polymarket's bid (the `K YES + PM NO` hedge leg).

The fill-versus-edge trade-off of §2.1 applies in each band.

## 3. Two-sided spread: fused across venues vs each venue alone

§1 pairs one venue's bid side with the other's ask side. Here both sides of the **same** book are paired: a maker quoting both sides earns `buy − sell` per round trip, where the resting YES ask fills at the taker-buy price and the resting YES bid at the taker-sell price. Three books:

- **PM only**: latest PM taker-buy print − latest PM taker-sell print; no maker fee.
- **K only**: the same on Kalshi, minus the Kalshi maker fee on both fills.
- **fused**: each side is the volume-weighted average of the latest PM and Kalshi prints on that side. The Kalshi maker fee is charged on Kalshi's volume share of each side. This is the spread of one maker quoting the same prices on both venues and filling in proportion to where the flow went.

An observation is a print on any leg of the series while all its legs are at most F old: two legs for a single venue, four for fused. The three series therefore have different populations; the counts show how different. The bucket is the YES price decile of (buy + sell) / 2. Same three figures as §1:

1. **count**: observations per bucket and per outcome, in thousands (team win = (home + away) / 2);
2. **% positive after maker fees**: the share with `buy − sell − maker fees > 0`;
3. **mean net spread** in cents, after maker fees.

Buckets with fewer than 10k observations per outcome are hatched. Positive here means the round trip beat the fees if both sides filled at those prices. Queue position and the adverse selection of the side that fills first are still unmeasured.

In [ ]:
from matplotlib.patches import Patch
from competition import GREEN, PURPLE
fs = pd.read_csv(OUT / 'fused_spread.csv')
fs['combo'] = fs.league + ' ' + fs.season.astype(str) + '/' + (fs.season % 100 + 1).astype(str).str.zfill(2)
fs = pd.concat([fs[(fs.combo == c) & (fs.phase == ph) & (fs.freshness == F)].assign(cell=cl) for cl, (c, ph, F) in CELLS.items()], ignore_index=True)
fs['sel'] = np.where(fs.outcome == 'draw', 'draw', 'team win')
sp = fs.groupby(['cell', 'sel', 'series', 'price_bucket'])[['n', 'pos', 'net_sum', 'gross_sum']].sum().reset_index()
sp['n_k'] = sp.n / 1000 / np.where(sp.sel == 'team win', 2, 1)   # per outcome
sp['pos_pct'] = 100 * sp.pos / sp.n
sp['net_c'] = 100 * sp.net_sum / sp.n
sp['gross_c'] = 100 * sp.gross_sum / sp.n
sp['thin'] = sp.n_k < MIN_N_K
SERIES = {'fused': (PURPLE, 'fused (PM + K)'), 'pm': (BLUE, 'PM only'), 'k': (GREEN, 'K only')}

def spread_panels(col, ylabel, title):
    fig, axes = grid(2, len(CELLS), w=4.6, h=3.0, sharex=True, sharey=True)
    x, w = np.arange(len(PB)), .27
    for i, sel in enumerate(['team win', 'draw']):
        for j, cl in enumerate(CELLS):
            ax = axes[i, j]
            for k, (s, (c, lab)) in enumerate(SERIES.items()):
                q = sp[(sp.cell == cl) & (sp.sel == sel) & (sp.series == s)].set_index('price_bucket').reindex(PB)
                for b, f in zip(ax.bar(x + (k - 1) * w, q[col], w, color=c, label=lab), q.thin.fillna(False).astype(bool)):
                    if f:
                        b.set_hatch('////'); b.set_alpha(.45)
            if col == 'net_c': ax.axhline(0, color='k', lw=.6)
            ax.set_xticks(x); ax.set_xticklabels(PB, rotation=45, fontsize=7)
        axes[i, 0].legend(handles=[Patch(color=col, label=lab) for col, lab in SERIES.values()], fontsize=7, loc='upper center', ncol=3)
    lo, hi = axes[0, 0].get_ylim()
    axes[0, 0].set_ylim(lo, hi + (hi - lo) * .25)   # headroom for the legend
    for ax in axes[-1]: ax.set_xlabel('YES price, (buy + sell) / 2')
    finish(fig, axes, ['team win', 'draw'], list(CELLS), ylabel, title)

spread_panels('n_k', 'observations per outcome (thousands)', 'Two-sided spread: observations by YES price level (hatched: < 10k)')
spread_panels('pos_pct', '% positive after maker fees', 'Two-sided spread buy − sell: % positive after maker fees (hatched: < 10k)')
spread_panels('net_c', 'mean net spread (cents)', 'Two-sided spread buy − sell: mean after maker fees (hatched: < 10k)')
tot = sp.groupby(['sel', 'series', 'cell'])[['n', 'pos', 'net_sum', 'gross_sum']].sum()
tot = tot.assign(pos_pct=100 * tot.pos / tot.n, net_c=100 * tot.net_sum / tot.n, gross_c=100 * tot.gross_sum / tot.n)
print('All buckets together (gross_c = before fees):')
display(tot[['pos_pct', 'gross_c', 'net_c']].unstack('cell').reindex(columns=list(CELLS), level=1).round(2))
for col, name in [('n_k', 'observations per outcome (thousands)'), ('pos_pct', '% positive after maker fees'), ('net_c', 'mean net spread (cents)')]:
    print(f'--- {name}')
    t = sp.pivot_table(index=['sel', 'series', 'cell'], columns='price_bucket', values=col).reindex(columns=PB)
    display(t.reindex(pd.MultiIndex.from_tuples([(s, r, cl) for s in ['team win', 'draw'] for r in SERIES for cl in CELLS])).round(2))

**Reading the current export.**

*A positive two-sided spread is nearly automatic; the size is what differs.* On a single venue the last taker buy prints above the last taker sell almost always, so the positive rate after maker fees is 86–90 % in the club cells and 95–100 % in the World Cup cells. Fused is lower, 84–93 %: it mixes the two venues' asynchronous prints. As with §1, the rate says little; the mean net spread carries the information.

*Polymarket's own book leaves the most per round trip, because its maker fee is 0.*
- PM only: 1.2–1.4 c net in the club cells, 0.75–0.87 c in the World Cup.
- Kalshi's gross spread is wider (club 1.5–1.8 c, World Cup 1.0 c), but two maker fees of up to 0.44 c each take most of it where the book is one tick wide. In the World Cup in play, K only nets 0.40–0.44 c overall and just 0.1–0.2 c at 30–60 c: one tick minus about 0.9 c of fees.
- Fused sits between the two (PL 1.0–1.25 c, World Cup in play 0.48–0.53 c). Quoting the same prices on both venues does not widen the spread. It only adds Kalshi's fee on the Kalshi share of the fills.

*Link to §1.* The cross-venue pair of §1 nets about the same as PM's own two-sided spread: World Cup in play, `PM YES + K NO` team win 0.72 c against PM only 0.75 c. PL: 1.20 c against 1.20 c. So pairing across venues is not a richer spread. What it could add is flow. Kalshi prints roughly twice as often as Polymarket (the counts: K only against PM only), so the Kalshi leg of a pair may fill faster than a second resting order on Polymarket's thinner side. That is a fill-rate question for the anchor and hedge sessions, not something the spread answers.

*Where Kalshi's own two-sided quoting is weakest:* mid prices in the World Cup, where one tick barely clears two maker fees. At 0–10 c and 90–100 c the Kalshi fee is small (0.09 c) and K only matches or beats PM only.

## 4. Team win: pair a PM YES bid with a PM YES ask or with a Kalshi NO bid?

Both series share one resting order, the **PM YES bid**, priced from the PM taker-sell print. They differ only in the other resting order:

- **PM only**: a PM YES ask, filled at the PM taker-buy print. Net = PM buy − PM sell; PM maker fee 0.
- **PM YES + K NO**: a Kalshi NO bid (a Kalshi YES ask), filled at the K taker-buy print. Net = K buy − PM sell − Kalshi maker fee. This is §1's maker/maker route.

**Paired snapshot, not a sequence.** An observation is a print on PM buy, PM sell or K buy while all three are at most F old (each cell's F), and both nets are computed on it from each leg's latest print. Neither leg has to fill first: within F the prints arrive in either order, and one print is reused until its leg prints again. This compares the two pairings at the same moment; what happens after one leg fills is left to the anchor and hedge sessions. The counts are therefore identical, and per observation
`(PM YES + K NO) − (PM only) = −(PM − K buy/buy gap) − Kalshi maker fee` (§2).
**Bucket** = the PM sell print's price decile, the shared PM bid reference. Team win only (home + away; counts per outcome = / 2).

Two figures:
1. **histogram** of the net edge after fees in right-closed 0.5 c bins: …, `(−0.5, 0]`, `(0, 0.5]`, `(0.5, 1]`, …, with open tails at ≤ −3 c and > 4 c. Every bin from `(0, 0.5]` up is net > 0. The bars show the share of each cell's observations.
2. **price level**, with rows following §1's three figures: observations per outcome (one bar, shared by both series), % positive after fees, and mean net edge. Buckets under 10k observations per outcome are hatched.

In [ ]:
pc = pd.read_csv(OUT / 'pm_vs_cross.csv')
pc['combo'] = pc.league + ' ' + pc.season.astype(str) + '/' + (pc.season % 100 + 1).astype(str).str.zfill(2)
pc = pd.concat([pc[(pc.combo == c) & (pc.phase == ph) & (pc.freshness == F)].assign(cell=cl) for cl, (c, ph, F) in CELLS.items()], ignore_index=True)
pc = pc[pc.outcome != 'draw']
ORANGE = '#ffa15a'
PAIR_SERIES = {'pm_only': (BLUE, 'PM only (PM YES ask)'), 'pm_yes_k_no': (ORANGE, 'PM YES + K NO (K NO bid)')}
BINS = list(range(-7, 9))   # k = (k/2, (k+1)/2] cents
def bin_label(k):
    if k == BINS[0]: return '≤−3'
    if k == BINS[-1]: return '>4'
    return f'({k / 2:g},{(k + 1) / 2:g}]'.replace('-', '−')
BIN_LABELS = [bin_label(k) for k in BINS]

# 1. histogram of the net edge per observation
h = pc.groupby(['cell', 'series', 'net_bin']).n.sum()
fig, axes = grid(1, len(CELLS), w=4.6, h=3.2, sharex=True, sharey=True)
x, w = np.arange(len(BINS)), .4
for j, cl in enumerate(CELLS):
    for k, (s, (c, lab)) in enumerate(PAIR_SERIES.items()):
        d = h.loc[(cl, s)].reindex(BINS, fill_value=0)
        axes[0, j].bar(x + (k - .5) * w, 100 * d / d.sum(), w, color=c, label=lab)
    axes[0, j].axvline(BINS.index(0) - .5, color='k', lw=.6, ls=':')   # left of it: net <= 0
    axes[0, j].set_xticks(x); axes[0, j].set_xticklabels(BIN_LABELS, rotation=60, fontsize=7)
    axes[0, j].set_xlabel('net edge after fees (cents)')
    axes[0, j].set_title(f'n={int(h.loc[(cl, "pm_only")].sum()):,}', fontsize=8)
axes[0, 0].legend(fontsize=7)
finish(fig, axes, ['team win'], list(CELLS), '% of paired observations', 'Net edge of the two pairings of a PM YES bid, paired observations (dotted: 0)')

# 2. price level: count, % positive, mean net
pp = pc.groupby(['cell', 'series', 'price_bucket'])[['n', 'pos', 'net_sum']].sum().reset_index()
pp['n_k'] = pp.n / 1000 / 2   # per team-win outcome
pp['pos_pct'] = 100 * pp.pos / pp.n
pp['net_c'] = 100 * pp.net_sum / pp.n
pp['thin'] = pp.n_k < MIN_N_K
ROWS4 = [('n_k', 'observations per outcome (thousands)'), ('pos_pct', '% positive after fees'), ('net_c', 'mean net edge (cents)')]
fig, axes = grid(len(ROWS4), len(CELLS), w=4.6, h=3.0, sharex=True, sharey='row')
for i, (col, _) in enumerate(ROWS4):
    for j, cl in enumerate(CELLS):
        ax = axes[i, j]
        series = {'pm_only': ('#9e9e9e', 'both series')} if col == 'n_k' else PAIR_SERIES
        ww = .8 if col == 'n_k' else w
        for k, (s, (c, lab)) in enumerate(series.items()):
            q = pp[(pp.cell == cl) & (pp.series == s)].set_index('price_bucket').reindex(PB)
            off = 0 if col == 'n_k' else (k - .5) * w
            for b, f in zip(ax.bar(x[:len(PB)] + off, q[col], ww, color=c, label=lab), q.thin.fillna(False).astype(bool)):
                if f:
                    b.set_hatch('////'); b.set_alpha(.45)
        if col == 'net_c': ax.axhline(0, color='k', lw=.6)
        ax.set_xticks(np.arange(len(PB))); ax.set_xticklabels(PB, rotation=45, fontsize=7)
    axes[i, 0].legend(fontsize=7, loc='upper center', ncol=2)
    lo, hi = axes[i, 0].get_ylim(); axes[i, 0].set_ylim(lo, hi + (hi - lo) * .25)   # headroom for the legend
for ax in axes[-1]: ax.set_xlabel('YES price of the PM sell print (PM YES bid reference)')
finish(fig, axes, [lab for _, lab in ROWS4], list(CELLS), '', 'Team win: PM only vs PM YES + K NO by price level, paired (hatched: < 10k per outcome)')

tot = pc.groupby(['series', 'cell'])[['n', 'pos', 'net_sum']].sum()
tot = tot.assign(n_k=tot.n / 2000, pos_pct=100 * tot.pos / tot.n, net_c=100 * tot.net_sum / tot.n)
print('All buckets together:')
display(tot[['n_k', 'pos_pct', 'net_c']].unstack('cell').reindex(columns=list(CELLS), level=1).round(2))
for col, name in ROWS4[1:]:
    print(f'--- {name}')
    t = pp.pivot_table(index=['series', 'cell'], columns='price_bucket', values=col).reindex(columns=PB)
    display(t.reindex(pd.MultiIndex.from_tuples([(s, cl) for s in PAIR_SERIES for cl in CELLS])).round(2))

**Reading the current export.**

*The same mean, a different shape.* On the same observations the two pairings net almost the same on average: 1.22 c against 1.25 c in the Premier League, 1.32 c against 1.26 c in the Champions League, 0.76 c against 0.73 c in the World Cup in play, and 0.87 c against 0.86 c in its last pregame hour. The distributions differ:

- **PM only** is concentrated on one tick. 59–84 % of observations fall in `(0.5, 1]`, and it is positive on 89–99 %.
- **PM YES + K NO** equals PM only plus `−(buy/buy gap) − Kalshi fee`. Because the buy/buy gap is mostly 0 or ±1 tick, the orange mass moves in two directions:
  - into `(1.5, 2]` (19–26 %): Kalshi's ask is a tick above Polymarket's, so the route earns two ticks minus the fee;
  - into `(−0.5, 0]` (11–16 %): Kalshi's ask is a tick below Polymarket's, so the route earns zero ticks minus the fee.

  When the two asks are level, it stays in `(0.5, 1]`: one tick minus the fee.

  Its positive rate is therefore lower, at 80–89 %.

*By price level* the difference is small and changes sign between cells, following §2.2's buy/buy gap:
- PL 60–90 c: the Kalshi pairing is better (1.37–1.46 c against 1.14–1.19 c), where Kalshi's ask sits highest above Polymarket's.
- UCL 0–10 c: the Kalshi pairing is better (1.44 c against 1.15 c).
- World Cup in play, 10–30 c: Polymarket is better (0.79–0.83 c against 0.59–0.66 c).
- Everywhere else, the two are within about 0.15 c.

**For a strategy.** Next to a resting PM YES bid, a PM YES ask carries the same expected edge as a Kalshi NO bid, with lower variance and a higher positive rate. The Kalshi order is worth it only where Kalshi's ask reliably sits a tick higher (PL above 60 c), or where it fills sooner. Because the snapshot edge is a draw, the choice turns on fill order and speed, which the next sessions measure from each PM bid fill forward.

## 5. Team win: does the PM bid or the Kalshi ask move first? Hayashi–Yoshida lead–lag

The two orders of §4's pairing are a resting PM YES bid, filled by PM taker sells, and a resting Kalshi YES ask (a NO bid), filled by Kalshi taker buys. If Kalshi's ask moves first, a PM bid fill tends to arrive after Kalshi has already repriced, and the pair's edge is gone when the bid fills. This section measures which series moves first and by how many seconds.

**Series.** Per fixture and team-win outcome (home, away), in play (0–105 min):
- `x` = PM taker-sell prints (per-second VWAP), the PM bid;
- `y` = Kalshi taker-buy prints, the Kalshi ask.

**Returns, not prints.** Each series' return is the price change between its consecutive prints, over its own interval. Nothing is synchronised and there is no freshness cut.

**Estimator** ([Hoffmann, Rosenbaum & Yoshida 2013](https://arxiv.org/abs/1303.4871)). The Hayashi–Yoshida covariance at lag θ multiplies every PM return by every Kalshi return whose interval, shifted θ seconds earlier, overlaps it:

`C(θ) = Σ_ij Δx_i Δy_j · 1{(t_{i−1}, t_i] ∩ (s_{j−1} − θ, s_j − θ] ≠ ∅}`

**θ > 0 means PM leads**: a Kalshi change is paired with the PM change θ seconds before it. The lead–lag estimate is the θ that maximises the correlation. Lags run from −30 to +30 s in 1 s steps. Timestamps are whole seconds, so θ = 0 mixes both orders within a second.

**Pooling.** C(θ), Σ Δx² and Σ Δy² are summed over all fixtures and outcomes of a group. Returns from different fixtures are never paired. `ρ(θ) = ΣC(θ) / √(ΣΔx² · ΣΔy²)`.

**Uncertainty.** 500 bootstrap resamples of fixtures give the bands and the interval of the peak lag.

**Summary numbers.**
- The lead–lag ratio `LLR = Σ_{θ>0} ρ² / Σ_{θ<0} ρ²` ([Huth & Abergel](https://arxiv.org/abs/1111.7103)): above 1 means PM leads on balance.
- `ρ(0)` and the peak ρ.

**Groups.**
- Every competition-season with at least 30 contributing fixtures (women's competitions excluded), each on its own.
- Then two pools: PL + UCL 2025/26, and all club competitions. The World Cup stays separate.

Goals are **not** treated separately. HY sums products of returns, so a few goal jumps (about 30 c on both venues) can carry much of each curve.

In [ ]:
hy = pd.read_parquet(OUT / 'hy_lead_lag.parquet')
hy = hy[hy.outcome != 'draw'].copy()   # team win: home + away
hy['combo'] = hy.league + ' ' + hy.season.astype(str) + '/' + (hy.season % 100 + 1).astype(str).str.zfill(2)
LAGS = np.sort(hy.lag.unique())
C = hy.groupby(['match_id', 'lag'])['cov'].sum().unstack('lag')[LAGS]   # fixture x lag, home + away summed
V = hy[hy.lag == LAGS[0]].groupby('match_id')[['var_pm', 'var_k', 'n_pm', 'n_k']].sum()
FX_COMBO = hy.drop_duplicates('match_id').set_index('match_id').combo
N_BOOT, MIN_FX = 500, 30
rng = np.random.default_rng(0)

def lead_lag(ids):
    # pooled HY correlation per lag, bootstrap band and peak-lag interval over fixtures
    c, v = C.loc[ids].to_numpy(), V.loc[ids]
    rho = c.sum(0) / np.sqrt(v.var_pm.sum() * v.var_k.sum())
    w = rng.multinomial(len(ids), np.full(len(ids), 1 / len(ids)), size=N_BOOT)   # resample fixtures
    boot = (w @ c) / np.sqrt((w @ v.var_pm.to_numpy()) * (w @ v.var_k.to_numpy()))[:, None]
    peak = LAGS[np.argmax(boot, axis=1)]
    pos, neg = LAGS > 0, LAGS < 0
    return dict(rho=rho, lo=np.percentile(boot, 2.5, 0), hi=np.percentile(boot, 97.5, 0), fixtures=len(ids),
                pm_prints_per_fx=v.n_pm.sum() / len(ids), k_prints_per_fx=v.n_k.sum() / len(ids),
                rho_0=rho[LAGS == 0][0], peak_rho=rho.max(), peak_lag=LAGS[np.argmax(rho)],
                peak_lag_lo=np.percentile(peak, 2.5), peak_lag_hi=np.percentile(peak, 97.5),
                llr=(rho[pos] ** 2).sum() / (rho[neg] ** 2).sum())

counts = FX_COMBO.value_counts()
combos = [c for c in counts.index if counts[c] >= MIN_FX and 'Women' not in c]
club = [c for c in combos if not c.startswith('World Cup')]
GROUPS = {c: [c] for c in combos}
GROUPS['PL + UCL 2025/26 (pooled)'] = ['Premier League 2025/26', 'UEFA Champions League 2025/26']
GROUPS['All club competitions (pooled)'] = club
res = {g: lead_lag(FX_COMBO.index[FX_COMBO.isin(cs)]) for g, cs in GROUPS.items()}

# 1. one panel per group
n = len(GROUPS); ncol = 4
fig, axes = grid(int(np.ceil(n / ncol)), ncol, w=4.6, h=3.0, sharex=True, sharey=True)
for ax, (g, r) in zip(axes.flat, res.items()):
    color = RED if g.startswith('World Cup') else ('#555555' if 'pooled' in g else BLUE)
    ax.fill_between(LAGS, r['lo'], r['hi'], color=color, alpha=.2, lw=0)
    ax.plot(LAGS, r['rho'], color=color, lw=1.5)
    ax.axvline(0, color='k', lw=.6); ax.axvline(r['peak_lag'], color=color, lw=.8, ls=':')
    ax.set_title(f"{g}\nn={r['fixtures']} fixtures, peak {r['peak_lag']:+d} s, LLR {r['llr']:.2f}", fontsize=8)
for ax in axes.flat[n:]: ax.set_visible(False)
for ax in axes[-1]: ax.set_xlabel('lag θ (s); > 0 = PM bid leads')
for ax in axes[:, 0]: ax.set_ylabel('HY correlation ρ(θ)')
fig.suptitle('Team win, in play: PM bid (taker sells) vs Kalshi ask (taker buys), Hayashi–Yoshida lead–lag (band: 95% fixture bootstrap)', fontsize=12, y=1.0)
fig.tight_layout(); plt.show()

# 2. shapes overlaid: each curve divided by its own peak
fig, ax = plt.subplots(figsize=(8, 4))
cmap = plt.get_cmap('tab10')
for k, g in enumerate(combos):
    r = res[g]
    ax.plot(LAGS, r['rho'] / r['peak_rho'], lw=2.2 if g.startswith('World Cup') else 1.2,
            ls='--' if g.startswith('World Cup') else '-', color=cmap(k), label=g)
ax.axvline(0, color='k', lw=.6); ax.set_xlabel('lag θ (s); > 0 = PM bid leads'); ax.set_ylabel('ρ(θ) / peak ρ')
ax.set_title('Curve shapes by competition (each divided by its peak)'); ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

tab = pd.DataFrame({g: {k: r[k] for k in ('fixtures', 'pm_prints_per_fx', 'k_prints_per_fx', 'rho_0', 'peak_rho', 'peak_lag',
                                         'peak_lag_lo', 'peak_lag_hi', 'llr')} for g, r in res.items()}).T
display(tab.round(3))
print('ρ(θ) at selected lags:')
sel = [-30, -20, -10, -5, -3, -2, -1, 0, 1, 2, 3, 5, 10, 20, 30]
display(pd.DataFrame({g: pd.Series(r['rho'], LAGS)[sel] for g, r in res.items()}).T.round(3))

**Reading the current export.**

*Kalshi's ask moves first, by about 3–6 s, in every competition.* No group peaks at θ ≥ 0. The peak lag, with its 95 % fixture-bootstrap interval:

| group | peak lag (s) | interval |
|---|---|---|
| World Cup 2026 | −3 | [−3, −3] |
| Premier League 25/26 | −5 | [−6, −3] |
| Champions League 25/26 | −5 | [−6, −3] |
| La Liga | −4 | [−4, −3] |
| Serie A | −6 | [−6, −4] |
| Ligue 1 | −6 | [−6, −5] |
| Bundesliga | −3 | [−6, −2] |
| Champions League 26/27 | −2 | [−3, −1] |
| PL + UCL 25/26, pooled | −5 | [−5, −3] |
| all club competitions, pooled | −5 | [−5, −3] |

*World Cup: a sharp peak.* ρ is 0.30 at −5 s, 0.59 at −3 s, 0.25 at 0 and 0.08 at +3 s, and about 0 beyond ±10 s. LLR is 0.05: almost all of the correlation sits on the side where Kalshi leads. The PM bid prints often enough to resolve this: about 1,385 prints per outcome in 105 minutes, one every 4.5 s on average.

*Club competitions: the same direction, a blunter shape.*
- Peak ρ is 0.47–0.69, but ρ at 0 is only a little lower (PL 0.58 at −5 s against 0.51 at 0).
- The curves are still at 0.14–0.36 at ±30 s.
- The PM bid prints far less often: 123–292 prints per outcome, one every 22–51 s on average (the table's `pm_prints_per_fx` counts both team-win outcomes). A long PM return overlaps Kalshi returns across many lags. So the shoulders mostly reflect sampling width, not a longer lead, and the peak location is the estimate.
- For the same reason, the club LLRs of 0.84–1.29 do not mean "no lead": the shoulders dominate them.
- The right shoulder (θ > 0) decays more slowly than the left in every club competition. With sparse PM prints and goals included, I do not read that as PM leading.

*Pooling.* The club competitions agree on the peak: −2 to −6 s with overlapping intervals. After dividing by the peak, the rising left side sits between −10 and −4 s in all of them. So pooling the clubs is reasonable for the lag. The level of ρ differs (Bundesliga 0.47, Champions League 26/27 0.69), and the Premier League has the narrowest shoulders. The World Cup is a different, much faster market.

This agrees with the VAR/VECM study, which found Kalshi leading in the World Cup. Here HY also finds Kalshi leading at the peak in club play, where the VECM study had Kalshi leading only in large moves.

**For a strategy.**
- **Leg 1 is adversely selected in time.** The resting PM YES bid tends to be hit a few seconds after Kalshi's ask has already moved. When Kalshi's ask falls, the PM bid is the next to fill, at a price that is now above Kalshi, so the Kalshi pairing is gone at the fill.
- **The same fact is a signal.** Kalshi's ask gives a 3–6 s warning for the PM bid: cancel or lower the PM YES bid when Kalshi's ask ticks down. The reaction budget is about 3 s in a World-Cup-speed market and about 5 s in club play, well within an engine that reads Kalshi's feed directly.
- **Next (anchor session):** at each PM bid fill, what did Kalshi's ask do in the preceding 10 s, and what was the pair's edge at the fill, conditional on it?

Caveats: goals are included and carry heavy weight; timestamps are whole seconds; these are trade prints, not quotes.


## 6. Team win: after one resting order fills, how soon does the other trade?

02.1 §2d's waiting curves, per resting order instead of per venue. Each resting order's fill is proxied by the taker print that would hit it:
- **PM YES bid**: PM taker sell;
- **PM YES ask**: PM taker buy;
- **Kalshi YES ask** (a Kalshi NO bid): Kalshi taker buy.

**Rate.** Take every source print second t, per fixture and team-win outcome, in play. The rate is the share of those seconds followed by a destination print strictly in (t, t + H], for H = 1 … 60 s on a linear axis.
- Same-second destination prints do not count as following. They stay in the denominator and are reported separately.
- A trigger is eligible for H only if t + H is still inside the in-play window (boundary censoring), so denominators shrink with H.
- Rates are pooled over trigger seconds across fixtures. Groups are the same as in §5.

Two figures:
1. **PM bid fills first**: PM YES bid → PM YES ask (blue), PM YES bid → Kalshi YES ask (orange), and PM YES bid → either ask (green), a trade on whichever of the two asks comes first. Blue and orange are the two ways to complete a pair once the PM bid has filled (§4); green is resting both and keeping whichever fills.
2. **The other order fills first**: PM YES ask → PM YES bid (blue), and Kalshi YES ask → PM YES bid (orange). Here the PM bid is the second fill.

A destination print is **any** trade on that side, at any price. It says when the other order could fill, not whether it would fill at a price that keeps the pair's edge.

In [ ]:
lw = pd.read_parquet(OUT / 'leg_waits.parquet')
lw = lw[lw.outcome != 'draw'].copy()   # team win: home + away
lw['combo'] = lw.league + ' ' + lw.season.astype(str) + '/' + (lw.season % 100 + 1).astype(str).str.zfill(2)
lw['pair'] = lw.source + ' -> ' + lw.dest
PAIR_NAMES = {'pm_bid -> pm_ask': 'PM bid fill → PM ask trade', 'pm_bid -> k_ask': 'PM bid fill → Kalshi ask trade',
              'pm_bid -> either_ask': 'PM bid fill → PM or Kalshi ask trade',
              'pm_ask -> pm_bid': 'PM ask fill → PM bid trade', 'k_ask -> pm_bid': 'Kalshi ask fill → PM bid trade'}
FIGS6 = {'PM YES bid fills first': [('pm_bid -> pm_ask', BLUE), ('pm_bid -> k_ask', ORANGE), ('pm_bid -> either_ask', GREEN)],
         'The other resting order fills first, PM YES bid second': [('pm_ask -> pm_bid', BLUE), ('k_ask -> pm_bid', ORANGE)]}
# one row per (fixture, outcome, pair) carries triggers / same_second; horizon rows carry eligible / followed
w6 = pd.concat([lw[lw.combo.isin(cs)].assign(group=g) for g, cs in GROUPS.items()], ignore_index=True)
wg = w6.groupby(['group', 'pair', 'horizon'])[['eligible', 'followed']].sum().reset_index()
wg['follow_pct'] = 100 * wg.followed / wg.eligible.replace(0, np.nan)
base = w6[w6.horizon == 1].groupby(['group', 'pair'])[['triggers', 'same_second']].sum()
base['same_second_pct'] = 100 * base.same_second / base.triggers
base['censored_pct_at_60'] = 100 * (1 - wg[wg.horizon == 60].set_index(['group', 'pair']).eligible / base.triggers)

for title, pairs in FIGS6.items():
    fig, axes = grid(int(np.ceil(len(GROUPS) / 4)), 4, w=4.6, h=3.0, sharex=True, sharey=True)
    for ax, g in zip(axes.flat, GROUPS):
        for p, color in pairs:
            d = wg[(wg.group == g) & (wg.pair == p)].sort_values('horizon')
            ax.plot(d.horizon, d.follow_pct, color=color, lw=1.5, marker='o', ms=2, label=PAIR_NAMES[p])
        ax.set_title(g, fontsize=8); ax.set_xlim(0, 60); ax.set_ylim(0, 100)
    for ax in axes.flat[len(GROUPS):]: ax.set_visible(False)
    for ax in axes[-1]: ax.set_xlabel('Forward horizon H (s)')
    for ax in axes[:, 0]: ax.set_ylabel('% with a strictly later\ndestination trade')
    axes[0, 0].legend(fontsize=7, loc='lower right')
    fig.suptitle(f'Team win, in play — {title}: H = 1 … 60 s', fontsize=12, y=1.0); fig.tight_layout(); plt.show()

print('Selected horizons (% of eligible trigger seconds followed):')
t6 = wg[wg.horizon.isin([1, 2, 3, 5, 10, 20, 30, 60])].pivot_table(index=['group', 'pair'], columns='horizon', values='follow_pct')
order = [(g, p) for g in GROUPS for p in PAIR_NAMES]
display(t6.reindex(order).rename(index=PAIR_NAMES, level=1).round(1))
print('Trigger seconds, same-second destination prints and censoring at H = 60 s:')
display(base.reindex(order).rename(index=PAIR_NAMES, level=1)[['triggers', 'same_second_pct', 'censored_pct_at_60']].round(1))

**Reading the current export.**

*Read the PM → PM curves from H = 2 s.* In the club competitions a PM trade follows a PM trade within 1 s on under 1 % of triggers, then on 13–28 % within 2 s. Polymarket timestamps are block times, about 2 s apart: in the early-season Premier League fixtures, no two consecutive PM prints are 1 s apart, against 12 % of Kalshi's. The World Cup's PM prints are 1 s apart more often, 17 %. Pairs with a Kalshi leg are not affected in this way.

*PM YES bid fills first: the Kalshi ask trades sooner than the PM ask, everywhere.*

| group | Kalshi ask within 5 / 10 / 60 s | PM ask within 5 / 10 / 60 s | same second, Kalshi / PM |
|---|---|---|---|
| Premier League | 65 / 81 / 98 % | 37 / 60 / 93 % | 26 / 22 % |
| Champions League 25/26 | 71 / 82 / 94 % | 45 / 66 / 92 % | 36 / 28 % |
| all clubs, pooled | 54 / 69 / 93 % | 33 / 53 / 88 % | 21 / 19 % |
| World Cup | 96 / 97 / 98 % | 84 / 94 / 99 % | 82 / 59 % |

The quietest leagues are slower but keep the same order: Ligue 1 is at 51 % against 41 % within 10 s. A same-second trade does not count as following, so the curves understate how often the second side trades almost at once.

*Resting both asks (green) completes the pair fastest.* A trade on whichever ask comes first follows a PM bid fill within 5 / 10 / 60 s on 74 / 88 / 99 % of triggers in the Premier League, against 65 / 81 / 98 % for the Kalshi ask alone. The pooled clubs are at 64 / 79 / 97 %, against 54 / 69 / 93 %, and the World Cup at 98 / 99 / 99 %. In the same second as the PM bid fill it trades on 34 % of triggers (clubs), against 21 % for the Kalshi ask alone.

The PM ask adds about 10 points at 5–10 s on top of the Kalshi ask. It is a real second exit, not a redundant one. The price of that speed is §4's double-fill risk.

*The other order fills first: the PM bid is the slow second leg.*
- After a Kalshi ask fill, a PM bid trade follows within 10 s on only 42 % of triggers (all clubs), 45 % (PL) and 82 % (World Cup).
- After a PM ask fill it follows on 48 %, 52 % and 86 %.

In the clubs, completing a pair that started on Kalshi takes about as long within 10 s as the fastest PM-first route takes within 3 s. It also starts far more often: 2.19 M Kalshi ask fills against 0.75 M PM bid fills in the club competitions, because Kalshi's flow is mostly buys.

*Consistent with §5.* Kalshi's ask moves 3–6 s before the PM bid. A PM bid fill is therefore usually followed quickly by Kalshi activity, while a Kalshi fill waits for PM to catch up.

**For a strategy.**
- **Rest both asks after the PM bid fills** if the double-fill risk can be sized: that completes the pair in 79–88 % of cases within 10 s in club play.
- **Anchor on the PM YES bid.** Once it fills, the Kalshi ask is the faster second leg: 69–81 % within 10 s in club play, against 53–60 % for the PM ask. Leg 1 is the scarce fill (§1), and leg 2 then has the most flow behind it.
- **Resting the Kalshi ask first is the slow order.** It fills often, but the PM bid may not trade for tens of seconds. By §5, when it does, it has usually followed Kalshi's move.
- **Not measured here: price.** Any destination trade counts. A resting order at a fixed price fills only on a trade at or through that price. The next step is to require the destination trade to clear the pair's edge, for example a Kalshi taker buy at or above the price that keeps `PM YES + K NO` positive after fees.


## 7. Team win: a replay simulation of the three resting orders

§1–6 described the pairs on reference prices. This section replays the tape against the three orders a real system would rest, using the recorded order books to place them and a queue model to fill them. It is the bridge to the real-money test: its parameters are the ones that test has to measure.

**Orders** (one team-win outcome, YES prices; order size and inventory limit L = 100 contracts). All three join the touch:

| order | price | fills on | target size |
|---|---|---|---|
| `pm_yes`: PM YES bid (leg 1) | b = PM best bid | PM taker sells at ≤ b | L − q |
| `pm_no`: PM YES ask = PM NO bid (leg 2) | a = PM best ask | PM taker buys at ≥ a | L + q |
| `k_no`: Kalshi YES ask = Kalshi NO bid (leg 2) | `k_price` = PM ask: a<br>`k_price` = Kalshi ask: Kalshi best ask | Kalshi taker buys at ≥ its price | L + q |

Inventory is `q = q_pm_yes − (q_pm_no + q_k_no)`, in contracts. A PM YES + PM NO pair merges into $1 at once (no fee). A PM YES + Kalshi NO pair is worth $1 at settlement, and the Kalshi leg pays the maker fee.

**Books and timing.**
- The books are the raw full-depth snapshots in `research/data/raw/orderbooks/soccer_1x2`: Polymarket from Predexon (about one every 3 s in club play), Kalshi from its websocket (under 1 s). Both exist for 1,141 fixtures; women's competitions are excluded. The Polymarket NO-token books are not needed: they mirror the YES books.
- The simulation runs in 1 s steps, the resolution of the trade tape. In each second, that second's taker trades hit the orders that were live from the previous second. Then, with the book known at the end of the second, orders are cancelled, resized or placed; they are live from the next second. That is about 1 s of latency.
- Two fills can land in the same second before the other ask is resized, which is the double-fill risk of §4.

**Fill model: a deterministic queue.**
- Each order joins behind `D = ρ × displayed size at its price`, and **only trades at that price** move it forward. Nobody ahead cancels.
- `ρ = 1` is therefore the worst case; `ρ < 1` stands for cancellations ahead of us; `ρ = 0` is the front of the queue, an optimistic bound.
- A trade **through** our price, i.e. at a worse price for the taker, means the level is gone: it fills min(remaining, volume).
- If our price improves the venue's book, nobody is ahead: D = 0. Only `k_no` can do this, when `k_price` = PM ask and that ask is below Kalshi's best ask.
- The tape is not changed by our orders.

**Sizing: each leg is a stack of child orders at one price** (up to 4). Every second the target size is recomputed:
- **Too big:** cancel the youngest children first. Kalshi can decrease an order in place and keep its queue position. Polymarket cannot amend an order, so that child is cancelled and the remainder re-posted at the back.
- **Too small:** top up with a new child at the back of the queue. Kalshi's amend-up would send the whole order to the back, so a separate order is better.
- Orders under 5 contracts are not placed.

**Start and end of a quote.** Each second, a leg is placed or kept only if:
- both books are valid (a best bid and ask, bid < ask) and at most 10 s old;
- the PM mid is 5–95 c;
- it is not the last 60 s of the 0–105 min window;
- for `k_no` also: its price > Kalshi's best bid (post-only), and its price − b − Kalshi fee > 0.

All of a leg's children are **cancelled** when its target price moves or it stops being allowed. It is re-placed at the next second where it is allowed.

**Cross-venue guard** (switch; replaces the earlier mid-based guard):
- rest `pm_yes` only while b < Kalshi's best ask;
- rest `pm_no` only while a > Kalshi's best bid (`k_no` always needs that).

Each side is guarded separately: when Kalshi drops, only the bid is pulled, and the asks stay.

**Grid.** ρ ∈ {1, 0.5, 0.25, 0} × `k_price` ∈ {PM ask, Kalshi ask} × guard ∈ {off, on}: 16 runs.

**Metrics.**
- **P&L per fixture:** both team-win outcomes; cash plus residual inventory marked to the last Kalshi mid before the final minute. It is split into **realized** (FIFO-matched YES/NO pairs) and **inventory** (the rest).
- **Contracts filled** and the **fill type**: at our level, our order improved the book, or traded through.
- **Markouts** vs the Kalshi mid 0 / 5 / 30 / 60 s after the fill second, in cents per contract, positive = good for us.
- **Inventory:** share of time |q| ≥ 1 contract, and double fills (max |q| > L).
- **Pair completion time:** the gap between a fill and the fill that matches it.

In [ ]:
sim = pd.read_parquet(OUT / 'mm_sim_summary.parquet'); fl7 = pd.read_parquet(OUT / 'mm_sim_fills.parquet'); pr7 = pd.read_parquet(OUT / 'mm_sim_pairs.parquet')
grid7 = pd.read_csv(OUT / 'mm_sim_grid.csv').set_index('param')
L7 = 100
KP = {0: 'K at PM ask', 1: 'K at K ask'}
grid7['label'] = [f"ρ={r.rho:g}, {KP[r.k_price]}, guard {'on' if r.guard else 'off'}" for r in grid7.itertuples()]
grp = lambda lg: np.where(lg.astype(str) == 'World Cup', 'World Cup', 'Clubs')
for d in (sim, fl7, pr7):
    d['group'] = grp(d.league)
for h in (0, 5, 30, 60):
    fl7[f'mo_{h}_x'] = 100 * fl7[f'mo_{h}'] * fl7.qty                 # cents x contracts
real = pr7.assign(r=pr7.pnl * pr7.qty).groupby(['param', 'match_id']).r.sum()
fx = sim.groupby(['param', 'group', 'match_id']).agg(value=('value', 'sum'), nz=('nz_q_sec', 'sum'), maxq=('max_abs_q', 'max')).reset_index()
fx['realized'] = real.reindex(pd.MultiIndex.from_frame(fx[['param', 'match_id']])).fillna(0).to_numpy()
fx['inventory'] = fx.value - fx.realized
rng7 = np.random.default_rng(0)
def boot_ci(x, n=1000):
    x = np.asarray(x); idx = rng7.integers(0, len(x), (n, len(x)))
    return np.percentile(x[idx].mean(1), [2.5, 97.5])

rows = []
for (p, g), d in fx.groupby(['param', 'group']):
    f = fl7[(fl7.param == p) & (fl7.group == g)]
    lo, hi = boot_ci(d.value)
    r = dict(param=p, group=g, fixtures=len(d), pnl_usd=d.value.mean(), ci_lo=lo, ci_hi=hi, realized_usd=d.realized.mean(),
             inventory_usd=d.inventory.mean(), q_nonzero_pct=100 * d.nz.sum() / (2 * len(d) * 6300),
             double_fill_pct=100 * (d.maxq > L7 + 1).mean(), losing_fx_pct=100 * (d.value < 0).mean(), p5_usd=d.value.quantile(.05),
             sd_usd=d.value.std(), fixtures_for_ci=int(np.ceil((1.96 * d.value.std() / d.value.mean()) ** 2)) if d.value.mean() > 0 else np.nan)
    for leg in ['pm_yes', 'pm_no', 'k_no']:
        x = f[f.leg == leg]
        r[f'{leg}_k'] = x.qty.sum() / len(d) / 1000
        r[f'{leg}_through_pct'] = 100 * x.qty[x.fill_type == 'through'].sum() / max(x.qty.sum(), 1e-9)
        for h in (0, 60):
            r[f'{leg}_mo{h}_c'] = x[f'mo_{h}_x'].sum() / max(x.qty.sum(), 1e-9)
    pp = pr7[(pr7.param == p) & (pr7.group == g)]
    r['k_exit_pct'] = 100 * pp.qty[pp.exit == 'k_no'].sum() / max(pp.qty.sum(), 1e-9)
    r['pair_median_s'] = pp.assign(w=pp.close_sec - pp.open_sec).sort_values('w').pipe(lambda z: z.w[z.qty.cumsum() >= z.qty.sum() / 2].iloc[0]) if len(pp) else np.nan
    rows.append(r)
res7 = pd.DataFrame(rows).join(grid7, on='param')

# 1. P&L per fixture, all runs: groups of four bars per ρ
RHOS = [1.0, 0.5, 0.25, 0.0]
VARIANTS = [(0, 0), (0, 1), (1, 0), (1, 1)]            # (k_price, guard)
fig, axes = grid(1, 2, w=7.5, h=3.8, sharey=False)
for ax, g in zip(axes[0], ['Clubs', 'World Cup']):
    d = res7[res7.group == g].set_index(['rho', 'k_price', 'guard'])
    for k, (kp, gd) in enumerate(VARIANTS):
        x = np.arange(len(RHOS)) + (k - 1.5) * .2
        y = d.loc[[(r_, kp, gd) for r_ in RHOS]]
        ax.bar(x, y.pnl_usd, .2, color=ORANGE if gd else BLUE, hatch='///' if kp else None, edgecolor='white',
               yerr=[y.pnl_usd - y.ci_lo, y.ci_hi - y.pnl_usd], capsize=1.5, label=f"{KP[kp]}, guard {'on' if gd else 'off'}")
        ax.scatter(x, y.realized_usd, color='k', marker='_', s=60, zorder=3, label='realized (matched pairs)' if k == 0 else None)
    ax.axhline(0, color='k', lw=.6)
    ax.set_xticks(range(len(RHOS))); ax.set_xticklabels([f'ρ = {r_:g}' for r_ in RHOS])
    ax.set_title(f"{g} ({int(d.fixtures.iloc[0])} fixtures)", fontsize=9); ax.set_ylabel('P&L per fixture ($, L = 100, 95% CI)')
axes[0, 0].legend(fontsize=7)
fig.suptitle('Simulated P&L per fixture (both team-win outcomes, in play); ρ = 1 is the worst-case queue', fontsize=12, y=1.0)
fig.tight_layout(); plt.show()

# 2-3. markouts and pair completion, clubs, ρ = 1 and 0.25
LINES = [(r_, kp, gd) for r_ in (1.0, 0.25) for kp in (0, 1) for gd in (0, 1)]
COL = {(1.0, 0): RED, (1.0, 1): PURPLE, (0.25, 0): BLUE, (0.25, 1): GREEN}
pid = {(r.rho, r.k_price, r.guard): p for p, r in grid7.iterrows()}
lab = lambda r_, kp, gd: f"ρ={r_:g}, {KP[kp]}, guard {'on' if gd else 'off'}"
fig, axes = grid(1, 3, w=4.6, h=3.4, sharey=True)
for ax, leg in zip(axes[0], ['pm_yes', 'pm_no', 'k_no']):
    for r_, kp, gd in LINES:
        f = fl7[(fl7.param == pid[r_, kp, gd]) & (fl7.group == 'Clubs') & (fl7.leg == leg)]
        ax.plot([0, 5, 30, 60], [f[f'mo_{h}_x'].sum() / f.qty.sum() for h in (0, 5, 30, 60)], marker='o', ms=3,
                color=COL[r_, kp], ls='--' if gd else '-', label=lab(r_, kp, gd))
    ax.axhline(0, color='k', lw=.6); ax.set_title(leg); ax.set_xlabel('seconds after the fill')
axes[0, 0].set_ylabel('markout vs Kalshi mid (c / contract)'); axes[0, 2].legend(fontsize=6, loc='upper right')
fig.suptitle('Clubs: markouts per contract by order (solid: guard off, dashed: on)', fontsize=12, y=1.0); fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7.5, 3.6))
for r_, kp, gd in LINES:
    pp = pr7[(pr7.param == pid[r_, kp, gd]) & (pr7.group == 'Clubs')].assign(w=lambda z: z.close_sec - z.open_sec).sort_values('w')
    ax.plot(pp.w.clip(lower=.5), pp.qty.cumsum() / pp.qty.sum(), color=COL[r_, kp], ls='--' if gd else '-', label=lab(r_, kp, gd))
ax.set_xscale('log'); ax.set_xlabel('seconds from a fill to its matching fill (0 shown at 0.5)'); ax.set_ylabel('share of matched contracts')
ax.set_title('Clubs: pair completion time'); ax.legend(fontsize=6); plt.tight_layout(); plt.show()

show = ['group', 'label', 'fixtures', 'pnl_usd', 'ci_lo', 'ci_hi', 'realized_usd', 'inventory_usd', 'pm_yes_k', 'pm_no_k', 'k_no_k',
        'pm_yes_through_pct', 'pm_no_through_pct', 'k_no_through_pct', 'pm_yes_mo0_c', 'pm_yes_mo60_c', 'pm_no_mo60_c', 'k_no_mo60_c',
        'k_exit_pct', 'pair_median_s', 'q_nonzero_pct', 'double_fill_pct', 'losing_fx_pct', 'p5_usd', 'sd_usd', 'fixtures_for_ci']
print('Per run (k = thousand contracts per fixture; markouts in cents per contract; P&L in $ per fixture):')
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(res7.sort_values(['group', 'rho', 'k_price', 'guard'], ascending=[True, False, True, True])[show].set_index(['group', 'label']).round(2))
print('Fill type shares, clubs, ρ = 1 and 0.25 (% of contracts):')
ft = fl7[(fl7.group == 'Clubs') & fl7.param.isin([pid[x] for x in LINES])]
display(ft.pivot_table(index=['param', 'leg'], columns='fill_type', values='qty', aggfunc='sum').pipe(lambda z: 100 * z.div(z.sum(axis=1), axis=0))
        .rename(index=grid7.label, level=0).round(1))
print('Per competition, worst-case queue ρ = 1 ($ per fixture):')
hp = grid7[grid7.rho == 1].index
cp = sim[sim.param.isin(hp)].groupby(['param', 'league', 'match_id']).value.sum().groupby(['param', 'league']).agg(['mean', 'size'])
display(cp.unstack('param').rename(columns=grid7.label, level=1).round(2))

**Reading the current export.**

*Positive even in the worst case.* With ρ = 1 (back of the displayed queue, nobody ahead cancels), every club variant makes money. Per fixture, L = 100, with 95 % interval:

| ρ = 1, clubs | guard off | guard on |
|---|---|---|
| `k_no` at PM ask | $9.1 [6.7, 11.3] | $11.8 [9.6, 14.3] |
| `k_no` at Kalshi ask | $13.0 [10.0, 15.9] | **$14.9 [12.0, 18.0]** |

Most of it is **realized** from matched YES/NO pairs: $12.2 of the $14.9, with $2.7 from inventory. Shorter queues scale it up: at ρ = 0.25 the best variant makes $25.9, and at ρ = 0 (front of the queue) $101. The World Cup makes $23–613, but that tournament is over.

*Rest `k_no` at Kalshi's best ask, not at the PM ask.* It is better in every club run: +$3–4 per fixture at ρ = 1, +$5–8 at ρ = 0.5–0.25, +$29–37 at ρ = 0.
- It gives up the "improved the book" fills, which were 51–54 % of `k_no` contracts at ρ = 1 in the PM-ask variant, and fills about as many contracts (0.80k against 0.84k per fixture).
- Those improved fills were not worth having: they sell a tick below Kalshi's ask. The `k_no` markout at 60 s is +0.17 to +0.21 c at Kalshi's ask against −0.08 to −0.10 c at the PM ask.
- So the most optimistic assumption of the previous run is gone, and the result got better, not worse.

*The cross-venue guard pays in club play.*
- **Effect:** +$1.9 to +$3.6 per fixture at ρ = 1–0.25 in all four club variants.
- **Better bid fills:** the PM bid's markout at the fill rises from 0.3–0.4 c to 1.3–1.5 c per contract (ρ = 1).
- **Fewer double fills:** 15 → 9 % of fixtures (ρ = 1, Kalshi-ask variant).
- **Where it doesn't pay:** the only club exception is ρ = 0 with `k_no` at the PM ask (−$3.0). At the World Cup it costs money in every run.

*Many fills come from the price running through the order.* At ρ = 1, 41–48 % of PM contracts filled are trade-throughs, and 55–56 % of `k_no` contracts at Kalshi's ask. Joining the back of a 2–7k queue mostly fills when the level is taken out. The markouts stay positive anyway.

*By competition* (ρ = 1, `k_no` at Kalshi ask, guard on): Bundesliga $25.0, La Liga $17.3, Premier League $16.4, Champions League $15.3, Serie A $10.8, Ligue 1 $3.7. Ligue 1 is the weakest in every variant.

*Inventory and risk* (ρ = 1, `k_no` at Kalshi ask, guard on):
- |q| ≥ 1 contract on 74 % of the in-play clock.
- The median matched contract waits 106 s for its pair.
- Standard deviation $49 per fixture; 38 % of fixtures lose; 5th percentile −$42.
- If the simulation were right, a live test would need about 42 fixtures (27 at ρ = 0.25) for its 95 % interval to exclude zero.

**Not modelled.**
- Our orders do not change the tape.
- Cancellations ahead of us exist only through ρ.
- PM books arrive about every 3 s. A websocket feed would re-quote faster; stale PM quotes get traded through here, so this is pessimistic for the PM legs.
- Latency is 1 s.
- Residual inventory is marked at the last Kalshi mid, not at settlement.
- No rebates.
- Team wins only.

**For a strategy.**
- Join the touch on all three orders, with `k_no` at Kalshi's best ask.
- Run the cross-venue guard in club play.
- Expect a positive but noisy result at the worst-case queue. How far above it the live result lands depends mainly on the real queue position (ρ), how fast double fills can be stopped, and how long inventory stays open.

### 7.1 Fewer orders: PM bid + Kalshi ask, or PM bid + PM ask

The same replay with one of the two asks switched off, to see what each exit adds:

| setup | resting orders | what it tests |
|---|---|---|
| **three orders** | `pm_yes` + `pm_no` + `k_no` | §7's design (repeated here as the reference) |
| **PM bid + Kalshi ask** | `pm_yes` + `k_no` | one exit: no double fills, one venue to cancel on after a fill |
| **PM bid + PM ask** | `pm_yes` + `pm_no` | a plain Polymarket market maker; with the guard on, Kalshi's book is still used as a signal |

**Settings.** Everything else is as in §7:
- `k_no` rests at Kalshi's best ask (§7's better variant);
- target sizes are L − q and L + q;
- ρ ∈ {1, 0.25} × guard ∈ {off, on}: 12 runs, written to `mm_sim_legs_*` by `mm_sim.py --variant legs`.

The three-order rows reproduce the matching §7 runs.

In [ ]:
sl = pd.read_parquet(OUT / 'mm_sim_legs_summary.parquet'); fll = pd.read_parquet(OUT / 'mm_sim_legs_fills.parquet'); prl = pd.read_parquet(OUT / 'mm_sim_legs_pairs.parquet')
gl = pd.read_csv(OUT / 'mm_sim_legs_grid.csv').set_index('param')
SETUP = {0: 'three orders', 1: 'PM bid + Kalshi ask', 2: 'PM bid + PM ask'}
gl['setup'] = gl.legs.map(SETUP)
for d in (sl, fll, prl):
    d['group'] = grp(d.league)
fll['mo0x'] = 100 * fll.mo_0 * fll.qty; fll['mo60x'] = 100 * fll.mo_60 * fll.qty
real_l = prl.assign(r=prl.pnl * prl.qty).groupby(['param', 'match_id']).r.sum()
fxl = sl.groupby(['param', 'group', 'match_id']).agg(value=('value', 'sum'), nz=('nz_q_sec', 'sum'), maxq=('max_abs_q', 'max')).reset_index()
fxl['realized'] = real_l.reindex(pd.MultiIndex.from_frame(fxl[['param', 'match_id']])).fillna(0).to_numpy()
rows = []
for (p, g), d in fxl.groupby(['param', 'group']):
    f, pp = fll[(fll.param == p) & (fll.group == g)], prl[(prl.param == p) & (prl.group == g)]
    lo, hi = boot_ci(d.value)
    bid = f[f.leg == 'pm_yes']
    rows.append(dict(param=p, group=g, fixtures=len(d), pnl_usd=d.value.mean(), ci_lo=lo, ci_hi=hi, realized_usd=d.realized.mean(),
                     inventory_usd=(d.value - d.realized).mean(), pm_yes_k=bid.qty.sum() / len(d) / 1000,
                     exit_k=f.qty[f.leg != 'pm_yes'].sum() / len(d) / 1000, pm_yes_mo0_c=bid.mo0x.sum() / bid.qty.sum(),
                     pm_yes_mo60_c=bid.mo60x.sum() / bid.qty.sum(),
                     pair_median_s=pp.assign(w=pp.close_sec - pp.open_sec).sort_values('w').pipe(lambda z: z.w[z.qty.cumsum() >= z.qty.sum() / 2].iloc[0]),
                     q_nonzero_pct=100 * d.nz.sum() / (2 * len(d) * 6300), double_fill_pct=100 * (d.maxq > L7 + 1).mean(),
                     losing_fx_pct=100 * (d.value < 0).mean(), p5_usd=d.value.quantile(.05), sd_usd=d.value.std(),
                     fixtures_for_ci=int(np.ceil((1.96 * d.value.std() / d.value.mean()) ** 2)) if d.value.mean() > 0 else np.nan))
resl = pd.DataFrame(rows).join(gl, on='param')

fig, axes = grid(1, 2, w=7.5, h=3.6, sharey=False)
COMBOS = [(1.0, 0), (1.0, 1), (0.25, 0), (0.25, 1)]
for ax, g in zip(axes[0], ['Clubs', 'World Cup']):
    d = resl[resl.group == g].set_index(['legs', 'rho', 'guard'])
    for k, (legs, color) in enumerate([(0, '#555555'), (1, ORANGE), (2, BLUE)]):
        x = np.arange(len(COMBOS)) + (k - 1) * .27
        y = d.loc[[(legs, r_, gd) for r_, gd in COMBOS]]
        ax.bar(x, y.pnl_usd, .27, color=color, yerr=[y.pnl_usd - y.ci_lo, y.ci_hi - y.pnl_usd], capsize=1.5, label=SETUP[legs])
        ax.scatter(x, y.realized_usd, color='k', marker='_', s=60, zorder=3, label='realized (matched pairs)' if k == 0 else None)
    ax.axhline(0, color='k', lw=.6)
    ax.set_xticks(range(len(COMBOS))); ax.set_xticklabels([f"ρ = {r_:g}\nguard {'on' if gd else 'off'}" for r_, gd in COMBOS])
    ax.set_title(f"{g} ({int(d.fixtures.iloc[0])} fixtures)", fontsize=9); ax.set_ylabel('P&L per fixture ($, L = 100, 95% CI)')
axes[0, 0].legend(fontsize=7)
fig.suptitle('Which orders to rest: P&L per fixture (team wins, in play; Kalshi order at Kalshi ask)', fontsize=12, y=1.0)
fig.tight_layout(); plt.show()

showl = ['fixtures', 'pnl_usd', 'ci_lo', 'ci_hi', 'realized_usd', 'inventory_usd', 'pm_yes_k', 'exit_k', 'pm_yes_mo0_c', 'pm_yes_mo60_c',
         'pair_median_s', 'q_nonzero_pct', 'double_fill_pct', 'losing_fx_pct', 'p5_usd', 'sd_usd', 'fixtures_for_ci']
print('Per run (k = thousand contracts per fixture; exit = pm_no + k_no; markouts in cents per contract; P&L in $ per fixture):')
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(resl.assign(run=[f"ρ={r.rho:g}, guard {'on' if r.guard else 'off'}" for r in resl.itertuples()])
            .sort_values(['group', 'rho', 'guard', 'legs'], ascending=[True, False, True, True]).set_index(['group', 'run', 'setup'])[showl].round(2))

**Reading the current export.**

*In club play, all three orders beat either pair, in every run.* Per fixture, L = 100:

| clubs | three orders | PM bid + Kalshi ask | PM bid + PM ask |
|---|---|---|---|
| ρ = 1, guard off | $13.0 | $8.0 | $10.1 |
| ρ = 1, guard on | **$14.9** | $10.5 | $11.8 |
| ρ = 0.25, guard off | $23.4 | $16.1 | $15.7 |
| ρ = 0.25, guard on | **$25.9** | $19.9 | $18.2 |

Dropping either ask costs $3–8 per fixture.

*Kalshi adds value twice.* A plain Polymarket market maker (PM bid + PM ask, guard off, no Kalshi at all) makes $10.1 at ρ = 1 and $15.7 at ρ = 0.25.
- **As a signal:** using Kalshi's book for the guard lifts it to $11.8 and $18.2.
- **As an exit:** adding the Kalshi order as well lifts it to $14.9 and $25.9, about +50–65 % over no Kalshi at all.

*The double-fill risk is worth taking.*
- The pairs never double-fill; the three orders do, in 9 % of club fixtures (ρ = 1, guard on).
- Yet the three orders earn more, with similar downside: 38 % of fixtures lose in each setup, and the 5th percentile is −$42 against −$47 and −$44.

*PM bid + Kalshi ask alone is the weakest club pair at ρ = 1.*
- Its inventory result is negative with the guard off (−$1.1 to −$1.2 per fixture).
- It needs the most fixtures to prove itself: 74 at ρ = 1 with the guard on, against 42 for three orders.
- It is simpler to run, but not better.

*PM bid + PM ask is the steadiest.* It has the lowest standard deviation ($40–45 per fixture) and needs the fewest fixtures at ρ = 0.25 with the guard (24). But its pairs take longest to complete: median 212 s at ρ = 1 with the guard, against 106 s for three orders.

*The guard helps every club setup:* +$2.4 and +$3.8 for PM bid + Kalshi ask, +$1.7 and +$2.5 for PM bid + PM ask.

*The World Cup is different.* PM bid + PM ask about matches three orders with the guard off: $41.2 against $41.4 at ρ = 1, $65.3 against $68.8 at ρ = 0.25. PM bid + Kalshi ask falls far behind ($8–32). There, the Kalshi exit adds little.

**For a strategy.** Keep all three orders for club play, with the guard on. If the live system has to start simpler, start with PM bid + PM ask plus the guard: it runs on one venue, has the lowest variance and the smallest sample needed at short queues, and the Kalshi exit can be added once double fills are under control.

### 7.2 A buffer on the guard: pull only on larger Kalshi moves

§7's guard pulls the PM bid as soon as Kalshi's best ask reaches it (Kalshi ask ≤ b). The idea tested here: a goal moves Kalshi several ticks at once, while a 1–2 tick move may be noise that comes back, so let Kalshi cross our price by a **buffer** before pulling.

**Rule** (buffer in ticks of 1 c):
- rest `pm_yes` only while b < Kalshi best ask + buffer;
- rest `pm_no` only while a > Kalshi best bid − buffer.

Buffer 0 is §7's guard. A larger buffer is a looser guard: it keeps the order up through small Kalshi moves and still pulls it when a jump arrives. Only a jump that lands within one second of the pull decision gets through, as it does at buffer 0.

**Settings.** Three orders, `k_no` at Kalshi's best ask, ρ ∈ {1, 0.25}. The guard is either off, or on with a buffer of 0, 1, 2 or 3 ticks: 10 runs, written to `mm_sim_buffer_*` by `mm_sim.py --variant buffer`.

In [ ]:
sb = pd.read_parquet(OUT / 'mm_sim_buffer_summary.parquet'); fb = pd.read_parquet(OUT / 'mm_sim_buffer_fills.parquet'); pb = pd.read_parquet(OUT / 'mm_sim_buffer_pairs.parquet')
gb = pd.read_csv(OUT / 'mm_sim_buffer_grid.csv').set_index('param')
gb['guard_label'] = [f'buffer {int(r.buffer)}' if r.guard else 'guard off' for r in gb.itertuples()]
for d in (sb, fb, pb):
    d['group'] = grp(d.league)
fb['mo0x'] = 100 * fb.mo_0 * fb.qty; fb['mo60x'] = 100 * fb.mo_60 * fb.qty
real_b = pb.assign(r=pb.pnl * pb.qty).groupby(['param', 'match_id']).r.sum()
fxb = sb.groupby(['param', 'group', 'match_id']).agg(value=('value', 'sum'), maxq=('max_abs_q', 'max'), pulls=('n_cancel_guard', 'sum')).reset_index()
fxb['realized'] = real_b.reindex(pd.MultiIndex.from_frame(fxb[['param', 'match_id']])).fillna(0).to_numpy()
rows = []
for (p, g), d in fxb.groupby(['param', 'group']):
    f = fb[(fb.param == p) & (fb.group == g)]
    bid, asks = f[f.leg == 'pm_yes'], f[f.leg != 'pm_yes']
    lo, hi = boot_ci(d.value)
    rows.append(dict(param=p, group=g, fixtures=len(d), pnl_usd=d.value.mean(), ci_lo=lo, ci_hi=hi, realized_usd=d.realized.mean(),
                     inventory_usd=(d.value - d.realized).mean(), guard_pulls=d.pulls.mean(), pm_yes_k=bid.qty.sum() / len(d) / 1000,
                     pm_yes_mo0_c=bid.mo0x.sum() / bid.qty.sum(), pm_yes_mo60_c=bid.mo60x.sum() / bid.qty.sum(),
                     asks_mo60_c=asks.mo60x.sum() / asks.qty.sum(), double_fill_pct=100 * (d.maxq > L7 + 1).mean(),
                     losing_fx_pct=100 * (d.value < 0).mean(), p5_usd=d.value.quantile(.05), sd_usd=d.value.std()))
resb = pd.DataFrame(rows).join(gb, on='param')

ORDER_B = ['guard off', 'buffer 0', 'buffer 1', 'buffer 2', 'buffer 3']
fig, axes = grid(1, 2, w=7.5, h=3.6, sharey=False)
for ax, g in zip(axes[0], ['Clubs', 'World Cup']):
    d = resb[resb.group == g]
    for k, (r_, color) in enumerate([(1.0, RED), (0.25, BLUE)]):
        y = d[d.rho == r_].set_index('guard_label').reindex(ORDER_B)
        x = np.arange(len(ORDER_B)) + (k - .5) * .38
        ax.bar(x, y.pnl_usd, .38, color=color, yerr=[y.pnl_usd - y.ci_lo, y.ci_hi - y.pnl_usd], capsize=1.5, label=f'ρ = {r_:g}')
        ax.scatter(x, y.realized_usd, color='k', marker='_', s=60, zorder=3, label='realized (matched pairs)' if k == 0 else None)
    ax.axhline(0, color='k', lw=.6); ax.set_xticks(range(len(ORDER_B))); ax.set_xticklabels(ORDER_B)
    ax.set_title(f"{g} ({int(d.fixtures.iloc[0])} fixtures)", fontsize=9); ax.set_ylabel('P&L per fixture ($, L = 100, 95% CI)')
axes[0, 0].legend(fontsize=7)
fig.suptitle('Guard buffer: P&L per fixture (three orders, Kalshi order at Kalshi ask; buffer in 1 c ticks)', fontsize=12, y=1.0)
fig.tight_layout(); plt.show()

showb = ['fixtures', 'pnl_usd', 'ci_lo', 'ci_hi', 'realized_usd', 'inventory_usd', 'guard_pulls', 'pm_yes_k', 'pm_yes_mo0_c', 'pm_yes_mo60_c',
         'asks_mo60_c', 'double_fill_pct', 'losing_fx_pct', 'p5_usd', 'sd_usd']
print('Per run (guard_pulls = guard cancels per fixture; k = thousand contracts per fixture; markouts in cents per contract):')
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(resb.assign(guard_label=pd.Categorical(resb.guard_label, ORDER_B, ordered=True))
            .sort_values(['group', 'rho', 'guard_label'], ascending=[True, False, True]).set_index(['group', 'rho', 'guard_label'])[showb].round(2))

**Reading the current export.**

*In club play the buffer is neutral.* P&L per fixture barely moves across buffers 0–3, and every interval overlaps:

| clubs | guard off | buffer 0 | buffer 1 | buffer 2 | buffer 3 |
|---|---|---|---|---|---|
| ρ = 1 | $13.0 | $14.9 | $14.7 | $14.3 | $14.2 |
| ρ = 0.25 | $23.4 | $25.9 | $26.0 | $25.6 | $25.4 |

What changes is the mix:
- **Fewer pulls:** 35 guard cancels per fixture at buffer 0, then 10, 4 and 2.
- **More bid fills:** 0.94k → 1.18k contracts per fixture at ρ = 1.
- **Each bid fill is worse:** its markout at the fill falls from 1.53 c to 1.15, 1.04 and 0.98 c per contract.
- **More double fills:** 9 → 13–15 % of fixtures.

So in club play the 1-tick crosses that buffer 0 reacts to are not pure noise: the fills they avoid are worse ones. Keeping the order up wins back about as much in volume as it loses in quality.

*At the World Cup the buffer removes the guard's cost.*
- Buffer 0 cost $9–13 per fixture there: $32.1 against $41.4 with the guard off at ρ = 1, $55.7 against $68.8 at ρ = 0.25.
- Buffer 2–3 is back at or slightly above guard off: $42.1–42.6 and $70.5–71.0.
- In that faster market, small crosses were mostly noise, and the guard only needs to catch jumps.

**For a strategy.**
- A buffer of **1 tick** is a robust default: club P&L equals buffer 0 ($14.7 against $14.9, $26.0 against $25.9) with 70 % fewer guard cancels.
- Buffer 1 recovers most of the World Cup loss ($39.1 against $32.1 at ρ = 1).
- Fewer cancels matter live more than here. The simulation charges nothing for a cancel beyond lost queue position, while a live system also pays in rate limits and Polymarket's cancel latency.
- The buffer is not a profit lever in club play; pick it for robustness.

### 7.3 Smaller orders: L = 100, 50, 20, 15 and 10

The §7 replay at smaller order sizes and inventory limits: L = 50 (`mm_sim.py --size 50`, written to `mm_sim_L50_*`) L = 20, 15 and 10 (`--size 20`, `mm_sim_L20_*`, and likewise for 15 and 10). Everything else is unchanged, except the minimum order: the L = 20, 15 and 10 runs post no Polymarket order or top-up under 5 contracts and no Kalshi order under 1 contract (the venues' minimums). The L = 100 and L = 50 runs used a 5-contract minimum on both venues. The table shows the worst-case queue (ρ = 1), all four variants, for club competitions and the World Cup.

- **P&L:** marked at the last Kalshi mid, as in §7. **Flatten** instead closes the leftover position at the end as a taker: it sells YES at the better best bid, or buys YES at the better best ask, fees included.
- **Double fills:** fixtures where |q| exceeded L.
- **Per contract of L:** P&L ÷ L.
- **Max loss:** the worst single fixture's P&L (both team-win outcomes), and the 5th percentile.
- **Cash needed per venue:** each second of a fixture, the cash tied up on that venue, summed over both team-win outcomes (one account per venue):
  - Polymarket: cash paid for positions, with every PM YES + PM NO pair merged back to $1 at once, plus the collateral of resting bids (b per contract) and of the part of the PM ask not covered by YES already held (1 − a per contract).
  - Kalshi: cash paid for NO (locked until settlement, fee included), plus the collateral of the resting NO bid (1 − a per contract).

  A fixture's requirement is the peak of that series. The second table gives its median, 90th and 95th percentiles and maximum across fixtures, for each venue and for both together. The two venues peak at different times, so the combined peak is below the sum of the two.

In [ ]:
rows, cap_rows = [], []
for Lx, prefix in ((100, 'mm_sim'), (50, 'mm_sim_L50'), (20, 'mm_sim_L20'), (15, 'mm_sim_L15'), (10, 'mm_sim_L10')):
    s_, f_, p_, c_ = (pd.read_parquet(OUT / f'{prefix}_{n}.parquet') for n in ('summary', 'fills', 'pairs', 'capital'))
    g_ = pd.read_csv(OUT / f'{prefix}_grid.csv').set_index('param')
    for grp_ in ('Clubs', 'World Cup'):
        for pid_, r_ in g_[g_.rho == 1.0].iterrows():
            pick_ = lambda d: d[(d.param == pid_) & ((d.league.astype(str) == 'World Cup') == (grp_ == 'World Cup'))]
            ss, ff, pp_, cc = pick_(s_), pick_(f_), pick_(p_), pick_(c_)
            fx_ = ss.groupby('match_id').agg(value=('value', 'sum'), flat=('value_flatten', 'sum'), maxq=('max_abs_q', 'max'))
            real_ = (pp_.pnl * pp_.qty).groupby(pp_.match_id).sum().reindex(fx_.index).fillna(0)
            lo_, hi_ = boot_ci(fx_.value)
            w_ = (pp_.close_sec - pp_.open_sec).to_numpy(); o_ = np.argsort(w_); cq_ = np.cumsum(pp_.qty.to_numpy()[o_])
            n_ = len(fx_)
            rows.append({'L': Lx, 'group': grp_, 'variant': f"{KP[r_.k_price]}, guard {'on' if r_.guard else 'off'}",
                         'P&L ($)': fx_.value.mean(), '95% CI': f'[{lo_:.1f}, {hi_:.1f}]', 'flatten ($)': fx_.flat.mean(),
                         'realized ($)': real_.mean(), 'inventory ($)': (fx_.value - real_).mean(),
                         'bid fills': ff.qty[ff.leg == 'pm_yes'].sum() / n_, 'PM ask fills': ff.qty[ff.leg == 'pm_no'].sum() / n_,
                         'K ask fills': ff.qty[ff.leg == 'k_no'].sum() / n_, 'losing %': 100 * (fx_.value < 0).mean(),
                         'SD ($)': fx_.value.std(), 'double fill %': 100 * (fx_.maxq > Lx + 1).mean(),
                         'median pair s': w_[o_][np.searchsorted(cq_, cq_[-1] / 2)], 'per contract of L': fx_.value.mean() / Lx,
                         'mean / SD': fx_.value.mean() / fx_.value.std(),
                         'p5 ($)': fx_.value.quantile(.05), 'max loss ($)': fx_.value.min(), 'worst fixture': fx_.value.idxmin()})
            cap_ = {'L': Lx, 'group': grp_, 'variant': rows[-1]['variant']}
            for col_, name_ in (('max_cap_pm', 'Polymarket'), ('max_cap_k', 'Kalshi'), ('max_cap_total', 'both')):
                v_ = cc[col_]
                cap_.update({f'{name_} median': v_.median(), f'{name_} p90': v_.quantile(.9), f'{name_} p95': v_.quantile(.95), f'{name_} max': v_.max()})
            cap_rows.append(cap_)
sz = pd.DataFrame(rows).sort_values(['group', 'variant', 'L'], ascending=[True, False, False]).set_index(['group', 'variant', 'L'])
print('Worst-case queue (ρ = 1): L = 100, 50, 20, 15 and 10 (fills in contracts per fixture):')
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(sz.round(2))
cap7 = pd.DataFrame(cap_rows).sort_values(['group', 'variant', 'L'], ascending=[True, False, False]).set_index(['group', 'variant', 'L'])
print("Cash needed within a fixture ($): each fixture's peak, summarised across fixtures")
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(cap7.round(0))

**Reading the current export.**

*Half the size keeps about 60 % of the P&L.* Clubs, `k_no` at Kalshi ask, guard on: $9.2 [7.6, 11.0] per fixture at L = 50, against $14.9 at L = 100. Fills fall only to about 60 %: 577 / 166 / 456 contracts against 940 / 272 / 762. A trade that reaches our price fills a larger share of a smaller order, so P&L per contract of L rises from 0.149 to 0.185. Pairs also complete faster, with a median of 88 s against 106 s.

*Risk falls faster than P&L.* The standard deviation per fixture drops from $49 to $28, and mean ÷ SD rises from 0.30 to 0.33. Losing fixtures drop from 38 % to 35 %. The double-fill rate relative to L is unchanged at 9 %.

*The ranking of variants is unchanged.* `k_no` at Kalshi ask with the guard on is best in the club competitions at both sizes. The World Cup scales the same way, to about 55 % of its L = 100 P&L.

**For the live test.** L = 50 needs about as many fixtures as L = 100 to separate the mean from zero, about 36 against 42, with half the capital at risk. A smaller order also makes the replay's "our orders do not change the tape" assumption more realistic.

*Max loss.* Clubs, `k_no` at Kalshi ask, guard on:

| L | 5th percentile | worst fixture |
|---|---|---|
| 100 | −$42 | −$113 |
| 50 | −$20 | −$53 |

The worst fixture is the same match in every club variant at a given L. Across the four club variants the worst fixture loses $112–160 at L = 100 and $53–78 at L = 50. The World Cup's worst is smaller ($53–86 at L = 100), because its pairs complete within seconds.

*Cash needed per fixture* (same variant; each fixture's peak, summarised across fixtures):

| L | venue | median | 95th percentile | max |
|---|---|---|---|---|
| 100 | Polymarket | $288 | $1,269 | $4,741 |
| 100 | Kalshi | $284 | $1,519 | $6,411 |
| 100 | both | $540 | $2,673 | $10,603 |
| 50 | Polymarket | $161 | $731 | $2,535 |
| 50 | Kalshi | $156 | $918 | $3,721 |
| 50 | both | $302 | $1,602 | $6,020 |

- The requirement grows with turnover, not with L alone. Every YES/NO pair completed across venues locks $1 until settlement: the PM YES on Polymarket, the Kalshi NO on Kalshi. So busy fixtures stack up locked pairs through the match.
- Only pairs completed on Polymarket alone are released at once, through a merge.
- The median fixture needs a few hundred dollars per venue. The tail is set by the busiest fixtures, and the World Cup needs 2–3 times as much.
- Halving L cuts the cash needed to 55–60 %, about the same factor as the P&L.

**For the live test.** At L = 50 the ~95 % budget to cover a club fixture is about $750 on Polymarket and $900 on Kalshi, and about $2,500 and $3,700 to cover the worst fixture in the sample. Several fixtures played at the same time add up. Merging PM YES + PM NO as soon as both are held, and preferring the Polymarket exit late in a match, both reduce the capital that stays locked until settlement.

*L = 20.* Same variant (clubs, `k_no` at Kalshi ask, guard on):
- **P&L:** $4.9 [4.1, 5.7] per fixture; 33 % of fixtures lose; 5th percentile −$8; worst fixture −$32.
- **Per contract of L:** 0.24 (0.19 at L = 50, 0.15 at L = 100). Mean ÷ SD is 0.37 (0.33, 0.30). Smaller orders keep a larger share of the flow that reaches their price.
- **Cash needed per fixture** (median / 90th / 95th percentile / max):
  - Polymarket: $73 / 252 / 344 / 1,057;
  - Kalshi: $73 / 308 / 450 / 1,664;
  - both: $144 / 538 / 770 / 2,668.
- **Sample size:** a live test at L = 20 needs about 28 club fixtures for a 95 % interval that excludes zero, if the replay is right.

The ranking of variants is the same at every size.

*L = 15 and 10.* The trend continues but flattens (same variant):

| L | P&L per fixture | losing | worst fixture | per contract of L | mean ÷ SD | cash, PM / Kalshi, 95th percentile |
|---|---|---|---|---|---|---|
| 15 | $3.9 [3.3, 4.6] | 32 % | −$25 | 0.26 | 0.38 | $274 / $358 |
| 10 | $2.9 [2.4, 3.3] | 31 % | −$17 | 0.29 | 0.39 | $195 / $254 |

Polymarket's 5-contract minimum is one reason the gain per contract slows at L = 10: a shortfall under 5 contracts is not topped up.

### 7.4 A cash cap per venue: L = 10 with $200 on Polymarket and $300 on Kalshi

§7.3 shows that at L = 10, about 5 % of club fixtures need more than $200 on Polymarket or $300 on Kalshi. This run caps the cash instead (`mm_sim.py --size 10 --cap-pm 200 --cap-k 300`, written to `mm_sim_L10_cap200_300_*`).

**Rule: stop opening, keep hedging.** Each second, the order sizes are cut to what the remaining cash can fund. Remaining cash is the cap minus cash already paid for positions.
- **Kalshi:** its cash funds only the NO bid.
- **Polymarket:** its cash is shared by the bid and by the part of the ask not covered by YES already held. Selling held YES needs no cash.
- **Priority:** whichever order reduces |q| is funded first; the order that opens new exposure gets what is left.

**Budget split.** The simulation runs each team-win outcome separately, so each outcome gets half the budget: $100 on Polymarket and $150 on Kalshi. A live account shares the cash between the two outcomes, so this split is slightly conservative.

Everything else is as in §7.3 at L = 10.

Two histograms follow the tables: the P&L per fixture for `k_no` at Kalshi ask with the guard on, without and with the cap. The first covers clubs and the World Cup at $1 bins, with the right tail clipped at the 99.5th percentile. The second covers clubs alone at $0.50 bins up to $25, with fixtures above $25 in the last bin.

In [ ]:
rows4 = []
for label_, prefix in (('no cap', 'mm_sim_L10'), ('cap $200 / $300', 'mm_sim_L10_cap200_300')):
    s_, c_ = (pd.read_parquet(OUT / f'{prefix}_{n}.parquet') for n in ('summary', 'capital'))
    g_ = pd.read_csv(OUT / f'{prefix}_grid.csv').set_index('param')
    for grp_ in ('Clubs', 'World Cup'):
        for pid_, r_ in g_[g_.rho == 1.0].iterrows():
            pick_ = lambda d: d[(d.param == pid_) & ((d.league.astype(str) == 'World Cup') == (grp_ == 'World Cup'))]
            fx_ = pick_(s_).groupby('match_id').agg(value=('value', 'sum'), maxq=('max_abs_q', 'max'))
            cc = pick_(c_)
            lo_, hi_ = boot_ci(fx_.value)
            q_ = lambda col: ' / '.join(f'{v:,.0f}' for v in cc[col].quantile([.5, .9, .95, 1.0]))
            rows4.append({'group': grp_, 'variant': f"{KP[r_.k_price]}, guard {'on' if r_.guard else 'off'}", 'run': label_,
                          'P&L ($)': fx_.value.mean(), '95% CI': f'[{lo_:.2f}, {hi_:.2f}]', 'losing %': 100 * (fx_.value < 0).mean(),
                          'p5 ($)': fx_.value.quantile(.05), 'max loss ($)': fx_.value.min(), 'SD ($)': fx_.value.std(),
                          'PM cash median / p90 / p95 / max': q_('max_cap_pm'), 'Kalshi cash median / p90 / p95 / max': q_('max_cap_k'),
                          'max |q|': fx_.maxq.max()})
cap4 = pd.DataFrame(rows4).sort_values(['group', 'variant', 'run'], ascending=[True, False, False]).set_index(['group', 'variant', 'run'])
print('L = 10, worst-case queue (ρ = 1): without and with the cash cap')
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(cap4.round(2))

# where the cap binds: fixtures that needed more than the cap without it
s0_, c0_ = pd.read_parquet(OUT / 'mm_sim_L10_summary.parquet'), pd.read_parquet(OUT / 'mm_sim_L10_capital.parquet')
s1_ = pd.read_parquet(OUT / 'mm_sim_L10_cap200_300_summary.parquet')
g0_ = pd.read_csv(OUT / 'mm_sim_L10_grid.csv').set_index('param')
bind = []
for pid_, r_ in g0_[g0_.rho == 1.0].iterrows():
    clubs_ = lambda d: d[(d.param == pid_) & (d.league.astype(str) != 'World Cup')]
    v0_, v1_ = clubs_(s0_).groupby('match_id').value.sum(), clubs_(s1_).groupby('match_id').value.sum()
    cc_ = clubs_(c0_).set_index('match_id')
    over_ = cc_.index[(cc_.max_cap_pm > 200) | (cc_.max_cap_k > 300)]
    bind.append({'variant': f"{KP[r_.k_price]}, guard {'on' if r_.guard else 'off'}", 'fixtures over the cap': len(over_),
                 'of fixtures': len(cc_), 'their mean P&L, no cap ($)': v0_.reindex(over_).mean(),
                 'their mean P&L, capped ($)': v1_.reindex(over_).mean(), 'their share of total P&L, no cap (%)': 100 * v0_.reindex(over_).sum() / v0_.sum()})
print('Clubs: the fixtures where the cap binds')
display(pd.DataFrame(bind).set_index('variant').round(2))

# P&L distribution per fixture: k_no at Kalshi ask, guard on, without and with the cap
RUNS4 = {'no cap': 'mm_sim_L10', 'cap $200 / $300': 'mm_sim_L10_cap200_300'}
dist4 = {}
for label_, prefix in RUNS4.items():
    s_ = pd.read_parquet(OUT / f'{prefix}_summary.parquet'); g_ = pd.read_csv(OUT / f'{prefix}_grid.csv').set_index('param')
    pid_ = g_[(g_.rho == 1.0) & (g_.k_price == 1) & (g_.guard == 1)].index[0]
    for grp_ in ('Clubs', 'World Cup'):
        d_ = s_[(s_.param == pid_) & ((s_.league.astype(str) == 'World Cup') == (grp_ == 'World Cup'))]
        dist4[(label_, grp_)] = d_.groupby('match_id').value.sum()
HCOL = {'no cap': BLUE, 'cap $200 / $300': ORANGE}
lab4 = lambda v, label_: f'{label_}: mean ${v.mean():.2f}, median ${v.median():.2f}, {100 * (v < 0).mean():.0f}% < 0'
with plt.rc_context({'text.parse_math': False}):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    for ax, grp_ in zip(axes, ('Clubs', 'World Cup')):
        allv = pd.concat([dist4[(l_, grp_)] for l_ in RUNS4])
        bins_ = np.arange(np.floor(allv.min()), np.ceil(allv.quantile(.995)) + 1, 1.0)
        for label_ in RUNS4:
            v_ = dist4[(label_, grp_)]
            ax.hist(v_.clip(upper=bins_[-1]), bins=bins_, alpha=.55, color=HCOL[label_], label=lab4(v_, label_))
            ax.axvline(v_.mean(), color=HCOL[label_], lw=1.5, ls='--')
        ax.axvline(0, color='k', lw=.8)
        ax.set_title(f'{grp_} ({len(dist4[("no cap", grp_)])} fixtures)')
        ax.set_xlabel('P&L per fixture ($), both team-win outcomes; right tail clipped at the 99.5th percentile')
        ax.set_ylabel('fixtures'); ax.legend(fontsize=8)
    fig.suptitle('P&L per fixture, L = 10, ρ = 1, Kalshi order at Kalshi ask, guard on (dashed: mean)', fontsize=12)
    fig.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(13, 4.6))
    bins_ = np.arange(np.floor(min(dist4[(l_, 'Clubs')].min() for l_ in RUNS4) * 2) / 2, 25.51, 0.5)   # last bin [25, 25.5) holds everything above 25
    for label_ in RUNS4:
        v_ = dist4[(label_, 'Clubs')]
        ax.hist(v_.clip(upper=25.25), bins=bins_, alpha=.55, color=HCOL[label_], label=lab4(v_, label_))
        ax.axvline(v_.mean(), color=HCOL[label_], lw=1.5, ls='--')
    ax.axvline(0, color='k', lw=.8); ax.set_xlim(bins_[0], 25.5)
    ax.set_xlabel('P&L per fixture ($), $0.50 bins; fixtures above $25 are in the last bin'); ax.set_ylabel('fixtures'); ax.legend(fontsize=8)
    ax.set_title(f'Clubs ({len(dist4[("no cap", "Clubs")])} fixtures): P&L per fixture, L = 10, ρ = 1, Kalshi order at Kalshi ask, guard on (dashed: mean)')
    plt.tight_layout(); plt.show()

qs_ = [0, .01, .05, .1, .25, .5, .75, .9, .95, .99, 1]
qt = pd.DataFrame({f'{grp_} · {label_}': dist4[(label_, grp_)].quantile(qs_) for grp_ in ('Clubs', 'World Cup') for label_ in RUNS4})
qt.index = ['min', 'p1', 'p5', 'p10', 'p25', 'median', 'p75', 'p90', 'p95', 'p99', 'max']
print('P&L per fixture ($): quantiles')
display(qt.round(2))
v_ = dist4[('no cap', 'Clubs')]
print(f'Clubs, no cap: share of total P&L from the top 10% of fixtures: {100 * v_.nlargest(len(v_) // 10).sum() / v_.sum():.1f}%; '
      f'fixtures above $25: {(v_ > 25).sum()} (no cap), {(dist4[("cap $200 / $300", "Clubs")] > 25).sum()} (capped)')

**Reading the current export.**

*The cap costs about 10 % of the P&L in club play.* `k_no` at Kalshi ask, guard on:

| | no cap | cap $200 / $300 |
|---|---|---|
| P&L per fixture | $2.86 [2.41, 3.31] | $2.57 [2.17, 3.01] |
| losing | 30.7 % | 31.2 % |
| 5th percentile / worst fixture | −$4.0 / −$16.9 | −$4.0 / −$16.9 |
| SD | $7.3 | $6.9 |
| PM cash: median / p90 / p95 / max | $39 / 139 / 195 / 556 | $39 / 110 / 122 / 200 |
| Kalshi cash: median / p90 / p95 / max | $40 / 171 / 254 / 897 | $40 / 156 / 197 / 300 |

The cash peaks stay within the cap. The other three club variants lose 10–11 % of their P&L, and their ranking is unchanged.

*The cap binds on the fixtures that earn the most.* Only 56 of 996 club fixtures needed more than $200 / $300 without the cap, but they carried 22 % of the total P&L. Capped, they still earn $7.3 per fixture on average, against $11.3 without the cap. Cash needed grows with the number of pairs completed, and so does P&L.

*Hedging is not affected.* The largest inventory is 30 contracts (3L) with or without the cap, and the worst fixture is unchanged. The priority rule keeps the reducing order funded.

*The World Cup is cut more,* by 15–28 %, because a World Cup fixture trades far more. That tournament is over.

**For the live test.** At L = 10, $200 on Polymarket and $300 on Kalshi per concurrent fixture keep about 90 % of the uncapped P&L. With a shared account across the two outcomes, as live, the loss should be smaller. Merging PM YES + PM NO at once and redeeming settled positions promptly both return cash to the budget.

*The distribution.* Clubs, `k_no` at Kalshi ask, guard on, no cap:
- **Most fixtures cluster just above zero.** The median is $1.1, and 31 % of fixtures lose, but losses stay small: 5th percentile −$4.0, worst −$16.9.
- **The profit comes from a long right tail.** The mean ($2.9) is well above the median. The 95th percentile is $13.9, the 99th $28.8, and the top 10 % of fixtures produce about 63 % of the total P&L. These are the busy fixtures with many completed pairs.
- **The cash cap trims the right tail and leaves the left one alone.** The 95th percentile falls to $11.8 and the 99th to $24.6; the 5th percentile and the worst fixture are unchanged.

For a live test, most single matches will show a small gain or a small loss; the edge appears only across many matches.

### 7.5 Where the P&L comes from, fixture by fixture (L = 10)

Why do fixtures differ, why do so many sit near zero, and where are the losses made? This section uses the §7.3 L = 10 run: ρ = 1, `k_no` at Kalshi ask, guard on, no cap, club competitions. The exports `mm_sim_L10_drivers.parquet` and `mm_sim_L10_jumps.parquet` come from [`mm_drivers.py`](../mm_drivers.py), which re-reads the books.

**Decomposition.** Let m(t) be the Kalshi mid at the start of second t. Each outcome's P&L splits into:
- **edge:** each fill's value against m at its second, net of the Kalshi fee. That is the spread captured.
- **inventory:** Σ q(t) · (m(t+1) − m(t)), the net YES position times the price change. The part in **jump seconds**, where |Δm| ≥ 3 c in one second, stands in for goals and other sudden repricings; there are no goal timestamps here.

Edge plus inventory reproduces the simulated P&L to within about $1 per fixture: the mark and the forward-filled mid differ slightly at the end.

**Drivers.** Per fixture:
- taker volume on each venue (team-win outcomes, in play);
- median displayed size at the best bid plus best ask on each venue;
- our filled contracts and quoted seconds;
- goals and jump seconds.

**Jumps.** For every jump second with a position, the table records the side of the position, the direction of the move, and which order, if any, filled in the 10 s before.

In [ ]:
drv = pd.read_parquet(OUT / 'mm_sim_L10_drivers.parquet'); jmp = pd.read_parquet(OUT / 'mm_sim_L10_jumps.parquet')
print(f"decomposition check: max |edge + inventory − P&L| = ${drv.check.abs().max():.2f} per fixture ({len(drv)} club fixtures)")

# 1. the spike near zero
near = []
for lo_, hi_ in ((-0.5, 0), (0, 0.5), (0.5, 1), (1, np.inf), (-np.inf, -0.5)):
    x_ = drv[(drv.pnl > lo_) & (drv.pnl <= hi_)]
    near.append({'P&L band ($)': f'({lo_}, {hi_}]', 'fixtures': len(x_), 'with zero fills': int((x_.fills == 0).sum()),
                 'median fills': x_.fills.median(), 'median quoted seconds (3 orders)': x_.quoted_s.median(),
                 'median PM taker volume': x_.pm_vol.median(), 'median Kalshi taker volume': x_.k_vol.median()})
print('Fixtures by P&L band:'); display(pd.DataFrame(near).set_index('P&L band ($)').round(0))

# 2. drivers
feat_ = ['pm_vol', 'k_vol', 'pm_depth', 'k_depth', 'fills', 'quoted_s', 'goals', 'n_jump']
print('Spearman correlation with P&L, edge and inventory P&L:')
display(drv[feat_ + ['pnl', 'edge', 'inv']].corr(method='spearman')[['pnl', 'edge', 'inv']].loc[feat_].round(2))
drv['volume quintile'] = pd.qcut(drv.pm_vol + drv.k_vol, 5, labels=['Q1 low', 'Q2', 'Q3', 'Q4', 'Q5 high'])
drv['depth tercile'] = pd.qcut(drv.pm_depth + drv.k_depth, 3, labels=['thin', 'mid', 'deep'])
print('By taker-volume quintile (means per fixture):')
display(drv.groupby('volume quintile', observed=True)[['pnl', 'edge', 'inv', 'fills', 'pm_depth', 'k_depth']].mean().round(2))
print('Mean P&L by volume quintile × touch-depth tercile:')
display(drv.pivot_table(index='volume quintile', columns='depth tercile', values='pnl', aggfunc='mean', observed=True).round(2))
gb_ = drv.groupby(drv.goals.clip(upper=6))
print('By goals (6 = 6 or more):'); display(gb_[['pnl', 'edge', 'inv', 'inv_jump']].mean().round(2).assign(fixtures=gb_.size()))

# 3. where the losses are made
dec = []
for name_, x_ in (('all', drv), ('losing', drv[drv.pnl < 0]), ('worst 5%', drv[drv.pnl <= drv.pnl.quantile(.05)])):
    dec.append({'fixtures': name_, 'n': len(x_), 'P&L': x_.pnl.mean(), 'edge': x_.edge.mean(), 'inventory': x_.inv.mean(),
                'inventory, jump seconds': x_.inv_jump.mean(), 'inventory, other': (x_.inv - x_.inv_jump).mean()})
print('P&L = edge + inventory (means per fixture, $):'); display(pd.DataFrame(dec).set_index('fixtures').round(2))
jmp['position and move'] = np.select([(jmp.dm > 0) & (jmp.q < 0), (jmp.dm < 0) & (jmp.q > 0), (jmp.dm > 0) & (jmp.q > 0), (jmp.dm < 0) & (jmp.q < 0)],
                                    ['short YES, price up', 'long YES, price down', 'long YES, price up', 'short YES, price down'], 'flat')
jmp['filled in the 10 s before'] = np.select([jmp.recent_k_ask > 0, jmp.recent_pm_ask > 0, jmp.recent_bid > 0],
                                            ['Kalshi ask', 'PM ask', 'PM bid'], 'nothing (position held earlier)')
print('Jump seconds with a position, all club fixtures (P&L in $):')
display(jmp.groupby('position and move').agg(seconds=('pnl', 'size'), pnl=('pnl', 'sum'), mean_abs_q=('q', lambda v: v.abs().mean())).round(1))
print('Losing jump seconds by what filled in the 10 s before (P&L in $):')
display(jmp[jmp.pnl < 0].groupby(['position and move', 'filled in the 10 s before']).agg(seconds=('pnl', 'size'), pnl=('pnl', 'sum')).round(1))
print('Worst five fixtures:')
display(drv.nsmallest(5, 'pnl')[['match_id', 'league', 'goals', 'pnl', 'edge', 'inv', 'inv_jump', 'n_jump', 'fills']].round(2).set_index('match_id'))

**Reading the current export.**

*The spike just below zero is fixtures that barely traded.*
- 117 club fixtures end between −$0.50 and $0, and 65 of them have no fill at all.
- The median quoted time across the three orders is only 154 s, against 3,886 s for fixtures above $1 and 2,291 s for those below −$0.50.
- Their markets did trade: median taker volume is comparable to the rest. Our orders were simply not live. Causes: a stale book (over 10 s old), the 5–95 c price band, the guard, or the back of the queue at ρ = 1.
- The fixtures between $0 and $0.50 did trade, but little (median 64 contracts).

*Volume drives the differences between fixtures, and depth matters only given volume.*
- **Correlation with P&L (Spearman):** our fills 0.46, quoted seconds 0.36, Kalshi taker volume 0.29, Polymarket taker volume 0.23. Touch depth alone is about zero (−0.05 and −0.02).
- **By volume:** from the lowest to the highest taker-volume quintile, P&L rises from $1.0 to $5.0 per fixture. Nearly all of the rise is edge ($2.4 to $7.2); inventory P&L is −$1.3 to −$2.2 in every quintile.
- **Depth, given volume:** within a volume quintile, deeper books earn less, because at ρ = 1 the whole displayed queue must trade first. In the top quintile: $7.3 (thin), $5.6 (mid), $4.1 (deep).
- **Goals:** they barely matter for P&L. More goals bring more edge and more inventory loss, which roughly cancel.

*Losses come from carrying inventory into jumps, not from being filled at the jump.*
- **Every fixture:** edge +$4.39 per fixture, inventory −$1.53, of which −$1.52 is in jump seconds. Outside jumps, inventory nets to about zero.
- **Losing fixtures:** edge is still +$1.90, but inventory is −$4.21 (−$3.35 in jump seconds). The five worst fixtures each lose $9–20 on inventory, mostly in jumps.
- **Short YES as the price jumps up** (a Kalshi NO or a PM ask filled, then that team scores) is the main loss: −$7,899 across the club fixtures. Of that, −$6,866 comes from positions held earlier and not yet paired; only −$899 from Kalshi asks filled in the 10 s before the jump.
- **Long YES as the price drops** (a PM bid filled, then the other team scores) costs −$1,868 in total. Bids filled in the 10 s before a drop lose only −$194, so the guard mostly prevents being hit just before a goal.
- **Jumps the other way pay back:** short YES as the price drops, and long YES as it rises, gain +$8,251. Net, jump seconds cost about $1,515, a third of the total edge of $4,373.

*Why mostly short YES.* Two asks (Polymarket and Kalshi) rest against one bid, each sized L + q. The ask side fills more often, so q is usually negative, and a short YES position loses when that team scores.

**For a strategy.** The lever is how long inventory is held, not the guard. Two changes are worth a replay:
- size the two asks so that together, rather than each, they cover L + q;
- close any unpaired position as a taker after a fixed time.

For the live test, record which of the quoting conditions keeps orders off the book, so zero-fill fixtures can be explained.

### 7.6 Per outcome or per fixture, strong or weak team (L = 10)

§7–7.5 run each team-win outcome on its own and add the two outcomes up per fixture. Nothing links them: no joint inventory limit, no hedge across outcomes, and §7.4's cash cap is split in half. This section looks at that sum, and at which of the two outcomes earns it.

**Run.** The §7.3 and §7.4 L = 10 runs: ρ = 1, `k_no` at Kalshi's best ask, guard on, club competitions; without and with the $200 / $300 cap.

**Strong and weak team.** The fused pre-game probability of [05](05_relative_strength.ipynb) (`fixture_probabilities`, `vwap_30m`: PM and Kalshi averaged, one venue when the other is missing). The team priced higher is **strong**, however small the gap (threshold 0), and the other is **weak**. Fixtures without a pre-game price are left out of the split.

**What it shows.**
1. P&L per fixture against P&L per outcome, as shares so the two can be compared.
2. P&L per outcome, strong against weak team, and their difference paired within the fixture.
3. The same at equal fills: quintiles of contracts filled per outcome.
4. For a small bankroll: 100 fixtures resampled 2,000 times, with the drawdown and the chance of being behind after 100. Fixtures are independent and do not overlap in time here.


In [ ]:
sys.path.insert(0, str(ROOT))
from research.soccer_1x2_analysis.relative_strength import fixture_probabilities
fq6 = fixture_probabilities(pd.read_parquet(ANALYSIS / 'outputs/pregame_prices/baskets.parquet'), 'vwap_30m').set_index('match_id')
FAV6 = pd.Series(np.select([fq6.q_home > fq6.q_away, fq6.q_away > fq6.q_home], ['home', 'away'], 'none'), index=fq6.index)
RUNS6 = {'no cap': 'mm_sim_L10', 'cap $200 / $300': 'mm_sim_L10_cap200_300'}
oc6 = {}                                               # one row per outcome: P&L, contracts filled, strong / weak
for label_, prefix in RUNS6.items():
    s_, f_ = pd.read_parquet(OUT / f'{prefix}_summary.parquet'), pd.read_parquet(OUT / f'{prefix}_fills.parquet')
    g_ = pd.read_csv(OUT / f'{prefix}_grid.csv').set_index('param')
    pid_ = g_[(g_.rho == 1.0) & (g_.k_price == 1) & (g_.guard == 1)].index[0]
    d_ = s_[(s_.param == pid_) & (s_.league.astype(str) != 'World Cup')].copy()
    d_['outcome'] = d_.outcome.astype(str)
    d_['value'] = d_.value.fillna(0)                   # no valid Kalshi mark and no position: worth 0, as in the fixture sums
    nf_ = f_[f_.param == pid_].astype({'outcome': str}).groupby(['match_id', 'outcome']).qty.sum()
    d_['fills'] = nf_.reindex(pd.MultiIndex.from_frame(d_[['match_id', 'outcome']])).fillna(0).to_numpy()
    fav_ = d_.match_id.map(FAV6).fillna('none')
    d_['role'] = np.where(fav_ == 'none', 'unlabelled', np.where(d_.outcome == fav_, 'strong', 'weak'))
    oc6[label_] = d_

def stats6(v):
    return {'n': len(v), 'mean ($)': v.mean(), 'median ($)': v.median(), 'SD ($)': v.std(), 'mean / SD': v.mean() / v.std(),
            'losing %': 100 * (v < 0).mean(), 'p5 ($)': v.quantile(.05), 'min ($)': v.min(), 'max ($)': v.max()}
lab6 = lambda name_, v: f'{name_} (n={len(v)}): mean ${v.mean():.2f}, median ${v.median():.2f}, {100 * (v < 0).mean():.0f}% < 0, p5 ${v.quantile(.05):.2f}'

# 1. per fixture (home + away) against per outcome, capped run
cap6 = oc6['cap $200 / $300']
fx6, po6 = cap6.groupby('match_id').value.sum(), cap6.value
w6 = cap6.pivot_table(index='match_id', columns='outcome', values='value').dropna()
with plt.rc_context({'text.parse_math': False}):
    fig, ax = plt.subplots(figsize=(13, 4.2))
    bins_ = np.arange(np.floor(min(fx6.min(), po6.min()) * 2) / 2, 25.51, 0.5)
    for v_, name_, c_ in ((fx6, 'per fixture (home + away)', BLUE), (po6, 'per outcome', ORANGE)):
        ax.hist(v_.clip(upper=25.25), bins=bins_, weights=np.full(len(v_), 100 / len(v_)), alpha=.55, color=c_, label=lab6(name_, v_))
        ax.axvline(v_.mean(), color=c_, lw=1.5, ls='--')
    ax.axvline(0, color='k', lw=.8); ax.set_xlim(bins_[0], 25.5); ax.legend(fontsize=8)
    ax.set_xlabel('P&L ($), $0.50 bins; above $25 in the last bin'); ax.set_ylabel('% of fixtures / outcomes')
    ax.set_title('Clubs, cap $200 / $300: P&L per fixture vs per outcome (L = 10, ρ = 1, Kalshi order at Kalshi ask, guard on; dashed: mean)')
    plt.tight_layout(); plt.show()
print(f'Correlation of home and away P&L within a fixture (capped, {len(w6)} fixtures with both): {w6.home.corr(w6.away):.2f}')

# 2. strong vs weak team, per outcome
with plt.rc_context({'text.parse_math': False}):
    fig, axes = plt.subplots(2, 1, figsize=(13, 7.6), sharex=True)
    for ax, (label_, d_) in zip(axes, oc6.items()):
        bins_ = np.arange(np.floor(d_.value.min() * 2) / 2, 25.51, 0.5)
        for role_, c_ in (('strong', RED), ('weak', BLUE)):
            v_ = d_.value[d_.role == role_]
            ax.hist(v_.clip(upper=25.25), bins=bins_, weights=np.full(len(v_), 100 / len(v_)), alpha=.5, color=c_, label=lab6(role_, v_))
            ax.axvline(v_.mean(), color=c_, ls='--', lw=1.5)
        ax.axvline(0, color='k', lw=.8); ax.set_xlim(bins_[0], 25.5); ax.set_ylabel('% of outcomes'); ax.legend(fontsize=8); ax.set_title(label_)
    axes[1].set_xlabel('P&L per outcome ($), $0.50 bins; above $25 in the last bin')
    fig.suptitle('Clubs, L = 10, ρ = 1, Kalshi order at Kalshi ask, guard on: strong vs weak team (pre-game vwap_30m, threshold 0; dashed: mean)', fontsize=11)
    fig.tight_layout(); plt.show()

rows6 = []
for label_, d_ in oc6.items():
    for name_, v_ in (('per fixture', d_.groupby('match_id').value.sum()), ('per outcome', d_.value),
                      ('strong', d_.value[d_.role == 'strong']), ('weak', d_.value[d_.role == 'weak'])):
        r_ = {'run': label_, 'unit': name_, **stats6(v_)}
        if name_ in ('strong', 'weak'):
            x_ = d_[d_.role == name_]
            r_.update({'zero-fill %': 100 * (x_.fills == 0).mean(), 'losing % if filled': 100 * (x_.value[x_.fills > 0] < 0).mean(),
                       'mean fills': x_.fills.mean()})
        rows6.append(r_)
print('P&L per fixture and per outcome ($):')
with pd.option_context('display.max_columns', None, 'display.width', 250):
    display(pd.DataFrame(rows6).set_index(['run', 'unit']).round(2))

pair6 = []
for label_, d_ in oc6.items():
    x_ = d_[d_.role != 'unlabelled'].pivot_table(index='match_id', columns='role', values='value').dropna()
    lo_, hi_ = boot_ci(x_.strong - x_.weak)
    pair6.append({'run': label_, 'fixtures': len(x_), 'strong - weak ($ per outcome)': (x_.strong - x_.weak).mean(), '95% CI': f'[{lo_:.2f}, {hi_:.2f}]'})
print('Strong minus weak, paired within the fixture:')
display(pd.DataFrame(pair6).set_index('run').round(2))

# 3. at equal fills: quintiles of contracts filled per outcome, no cap
d_ = oc6['no cap'][oc6['no cap'].role != 'unlabelled'].copy()
d_['fill quintile'] = pd.qcut(d_.fills.rank(method='first'), 5, labels=['F1 low', 'F2', 'F3', 'F4', 'F5 high'])
print('No cap, by fill quintile (per outcome):')
display(d_.groupby(['fill quintile', 'role'], observed=True).agg(
    n=('value', 'size'), mean_fills=('fills', 'mean'), mean_usd=('value', 'mean'), median_usd=('value', 'median'),
    sd_usd=('value', 'std'), losing_pct=('value', lambda v: 100 * (v < 0).mean())).unstack('role').round(2))

# 4. small bankroll: 100 fixtures drawn with replacement, 2,000 paths
rng6 = np.random.default_rng(0)
dd6 = []
for label_, d_ in oc6.items():
    for name_, v_ in (('strong only', d_.value[d_.role == 'strong']), ('weak only', d_.value[d_.role == 'weak']),
                      ('both outcomes', d_.groupby('match_id').value.sum())):
        p_ = rng6.choice(v_.to_numpy(), (2000, 100)).cumsum(1)
        dd_ = (np.maximum.accumulate(np.maximum(p_, 0), 1) - p_).max(1)
        dd6.append({'run': label_, 'traded': name_, 'mean ($ per fixture)': v_.mean(), 'mean / SD': v_.mean() / v_.std(),
                    'P&L after 100, median ($)': np.median(p_[:, -1]), 'P(P&L after 100 < 0) %': 100 * (p_[:, -1] < 0).mean(),
                    'max drawdown, median ($)': np.median(dd_), 'max drawdown, p95 ($)': np.percentile(dd_, 95)})
print('100 fixtures in a row (resampled; fixtures independent, no overlap in time):')
display(pd.DataFrame(dd6).set_index(['run', 'traded']).round(2))

**Reading the current export.**

*Adding the two outcomes up is already the fixture's P&L.* The replay is deterministic and our orders do not change the tape. So running each outcome on its own and summing gives exactly what a joint run with no rule linking the outcomes would give. A goal moves both outcomes on the same price path, so its effect on both is already in the sum. A joint run differs only when a rule uses both outcomes:
- **A shared cash cap:** it binds only on the busy fixtures, so it would move the right tail and leave the left one alone, as §7.4's cap did.
- **A joint inventory limit or a hedge across outcomes:** it changes the orders themselves, so it needs a new replay.

*The two outcomes do not offset each other.* Their P&L within a fixture is **positively** correlated (+0.16, capped run). What they share is the fixture's volume: busy fixtures earn on both outcomes, quiet ones are near zero on both. Capped, per fixture against per outcome:
- the mean doubles ($2.57 against $1.31);
- the left tail barely moves: 5th percentile −$4.0 in both, worst −$16.9 against −$15.8;
- the spike at zero is lower per fixture, because one outcome without fills rarely means a fixture without fills.

A shared cap is therefore likely to win back little: the two outcomes tend to need cash at the same time.

*Strong against weak team* (no cap):

| | mean | median | losing | losing, if filled | no fill | mean fills | SD | worst |
|---|---|---|---|---|---|---|---|---|
| strong | $1.76 | $0.45 | 33 % | 36 % | 10 % | 215 | $5.5 | −$15.8 |
| weak | $1.19 | $0.51 | 23 % | 28 % | 17 % | 126 | $4.1 | −$11.8 |

- **Paired within the fixture,** the strong team earns $0.55 more per outcome, 95 % interval [0.13, 0.95]. With the cap the gap is $0.35, [−0.02, 0.79]: the cap trims the strong team's busiest fixtures.
- **The weak team's lower losing share is partly fixtures where we never traded.** 17 % of weak outcomes have no fill at all; among filled outcomes it is 28 % against 36 %.

*At equal fills the strong team is worse, not better.* Within a quintile of contracts filled, the SD is about the same for both teams, so the extra variance of the strong team comes from its extra fills. The mean does not follow:

| fills per outcome | mean, strong / weak | losing, strong / weak |
|---|---|---|
| F2 (~27) | $0.00 / $0.78 | 49 % / 25 % |
| F3 (~86) | $0.50 / $0.95 | 44 % / 32 % |
| F4 (~187) | $1.11 / $1.70 | 41 % / 29 % |
| F5 (~545) | $5.06 / $3.71 | 21 % / 24 % |

The strong team's higher total comes from trading more, not from earning more per contract. Only in the busiest quintile does it pull ahead. A likely reason, not tested here: §7.5 found the position is mostly short YES, because two asks rest against one bid, and the stronger team scores more often, which is the jump that hurts a short YES. Splitting §7.5's edge / inventory decomposition by team would test it.

*For a small bankroll, the losing share matters less than it looks.* 100 fixtures resampled 2,000 times (no cap; capped in the table):

| traded | mean per fixture | mean / SD | max drawdown, median / p95 | behind after 100 |
|---|---|---|---|---|
| strong only | $1.76 | 0.32 | $13 / $24 | 0 % |
| weak only | $1.19 | 0.29 | $12 / $22 | 0.05 % |
| both outcomes | $2.86 | 0.39 | $12 / $23 | 0 % |

- A 95th-percentile drawdown of about $23 is under 5 % of a $200 + $300 budget. Loss size is not what a small account runs out of; cash locked in open positions is (§7.3–7.4).
- Trading the weak team alone barely lowers the drawdown ($1 less) and gives up 60 % of the P&L, with a lower mean / SD.
- One fixture in three loses. A short run of losses at the start of a live test is expected and is not evidence against the strategy: §7.3 needs about 30–40 fixtures for an interval that excludes zero.

**For a strategy.**
- Keep both outcomes, and do not filter by team strength.
- The strong team's lower P&L per contract at equal fills points at inventory held into the favourite's goals. That is the same lever as §7.5: shorter holding times, or asks sized to cover L + q together rather than each.
- Simulation limits as in §7. Fixtures here are independent and sequential; several fixtures played at the same time add up in cash and drawdown.

## What the real-money test should record

The first live test runs the §7.4 configuration: L = 10, `k_no` at Kalshi's best ask, the cross-venue guard, and a cash cap of $200 on Polymarket and $300 on Kalshi. The list is built to keep storage small. Full-depth order-book deltas from both websockets are by far the largest stream, and the analysis needs them only around our fills. Every event carries **exchange and local timestamps in ms**, plus the clock offset.

**1. Keep in full (small and irreplaceable)**
- **Our orders and fills.** Every decision, with its order (`pm_yes`, `pm_no`, `k_no`), action (new, top-up, shrink, cancel, re-post) and reason (touch moved, guard, cap, stale book, price band, end of window, below minimum size). Then the request, the acknowledgement, rejects with their reason, cancels and decreases, and fills (trade id, price, size, maker flag, fee charged).
- **One state row per decision.** Best bid and ask with their sizes on both venues, q per outcome, holdings per venue, cash per venue (available, reserved, locked), headroom under the cap, guard state, and book age. This explains every action without the full book.
- **Queue numbers per resting order, computed live:**
  - the displayed size at our price when the order is acknowledged;
  - the volume traded at our price before our fill or cancel;
  - the cancellations ahead of us.

  These give the **realized ρ** and the fill type (at our level or traded through) without storing book deltas.
- **Kalshi queue position from the exchange.** Poll `GET /portfolio/orders/queue_positions` (one call per event covers all our resting `k_no` orders) every few seconds and after every trade at our price, and record `queue_position_fp` per order: the exchange's own count of contracts ahead of us. It checks the queue numbers above, and so the realized ρ, on Kalshi. Polymarket has no equivalent.
- **Fill windows at full resolution.** For every fill on either venue, keep the raw websocket book deltas and public trades of that outcome on both venues from 5 s before to 30 s after the fill. The 1 s snapshots below are too coarse to show who moves after a fill, how fast the other venue reprices, and whether the other leg could have been pulled in time.
- **Public trades** on the markets traded: price, size, taker side, trade id. They are needed for markouts and the replay.
- **Match events:** actual kickoff, half-time, full-time, goals and red cards, with times.
- **Cash and settlement:**
  - balances per venue at each fixture's start and end;
  - Polymarket merges (time, gas);
  - redemptions;
  - Kalshi settlement credits;
  - fees billed, against the model.
- **Configuration and operations:**
  - per fixture: L, caps, guard and buffer, minimum sizes, and the code commit;
  - websocket disconnects and gaps, API errors, rate limits, kill-switch triggers.

**2. Keep, downsampled**
- **Order books: one snapshot per second** of the best price and the next 4 levels on each side, for each traded market, in play only. That is exactly what `mm_sim.py` reads (best + 3 ticks per second), so §7 can be re-run on the live fixtures and compared order by order. Roughly 1 MB per fixture as compressed Parquet, against tens to hundreds of MB for full deltas.
- **Per-second state:** written only when something changes.

**3. Do not keep permanently**
- **Raw websocket messages and full-depth deltas:** keep a rolling 24–48 h buffer for debugging, then delete everything outside the fill windows above.
- **Books of markets not traded:** keep at most a few fixtures as a control.
- **Derived metrics:** markouts at 1 / 5 / 30 / 60 s, edge against the mid, the inventory split into goal and non-goal seconds (§7.5), pair times, quoted seconds lost per reason. Recompute them offline from the kept data.

**Housekeeping.** Write daily compressed Parquet (zstd), one folder per fixture, and delete the raw buffer once the day's files are verified. Storage then grows by a few MB per fixture.

**Analysis plan**
- After each batch of fixtures, re-run `mm_sim.py` on the recorded 1 s books of the same fixtures and compare fills, fill types, realized ρ, markouts and P&L order by order.
- At L = 10 with the cap, expect about 25–30 club fixtures before a 95 % interval on the mean P&L excludes zero, if the replay is right.
- Change one setting at a time (for example the guard), by fixture rather than inside a match.